Generate SIMULATED ClinVar 2026 dataset — 1804 missense SNVs across 5 genes

## ⚠️ Known Warning: LightGBM joblib CPU count

During model training, LightGBM emits the following non-fatal warning:

```
UserWarning: Could not find the number of physical cores for the following reason:
'cpu_count' not found in module 'joblib.externals.loky.backend.context'
```

**Impact:** LightGBM likely ran single-threaded (n_jobs=1 effective). This does not affect model correctness or AUC values — only training speed. The warning arises from a version mismatch between LightGBM and the installed joblib backend in this environment.

**Resolution:** Not required for reproducibility. To suppress: `import warnings; warnings.filterwarnings('ignore', category=UserWarning, module='lightgbm')`.


In [1]:

import numpy as np
import pandas as pd
from scipy.stats import beta, truncnorm

np.random.seed(42)

# ── Gene-level config ──────────────────────────────────────────────────────────
gene_config = {
    "BRCA1": {"n": 380, "path_frac": 0.38, "chrom": "17", "start": 43044295},
    "BRCA2": {"n": 360, "path_frac": 0.35, "chrom": "13", "start": 32315086},
    "TP53":  {"n": 320, "path_frac": 0.40, "chrom": "17", "start": 7661779},
    "TREM2": {"n": 280, "path_frac": 0.25, "chrom": "6",  "start": 41161504},
    "EGFR":  {"n": 240, "path_frac": 0.28, "chrom": "7",  "start": 55019017},
    "ALK":   {"n": 224, "path_frac": 0.22, "chrom": "2",  "start": 29192774},
}

# Amino acids for variant generation
AA = list("ACDEFGHIKLMNPQRSTVWY")

def sample_features(n, is_pathogenic):
    """Sample realistic feature values conditioned on pathogenicity."""
    if is_pathogenic:
        sift       = np.random.beta(1.5, 6, n)          # low = damaging
        polyphen   = np.random.beta(6, 1.5, n)          # high = damaging
        gnomad_af  = np.random.beta(0.3, 30, n)         # rare
        cadd       = np.random.normal(28, 6, n).clip(10, 50)
        revel      = np.random.beta(5, 1.5, n)          # high = pathogenic
        am_score   = np.random.beta(6, 1.5, n)          # AlphaMissense
        in_domain  = np.random.binomial(1, 0.72, n)
        conserved  = np.random.binomial(1, 0.80, n)
        splice_prox= np.random.binomial(1, 0.15, n)
    else:
        sift       = np.random.beta(5, 1.5, n)          # high = tolerated
        polyphen   = np.random.beta(1.5, 6, n)          # low = benign
        gnomad_af  = np.random.beta(1.5, 5, n)          # more common
        cadd       = np.random.normal(12, 5, n).clip(1, 35)
        revel      = np.random.beta(1.5, 5, n)
        am_score   = np.random.beta(1.5, 6, n)
        in_domain  = np.random.binomial(1, 0.35, n)
        conserved  = np.random.binomial(1, 0.40, n)
        splice_prox= np.random.binomial(1, 0.05, n)
    return sift, polyphen, gnomad_af, cadd, revel, am_score, in_domain, conserved, splice_prox

rows = []
variant_id = 1

for gene, cfg in gene_config.items():
    n_total = cfg["n"]
    n_path  = int(round(n_total * cfg["path_frac"]))
    n_ben   = n_total - n_path

    for label, n in [(1, n_path), (0, n_ben)]:
        sift, pp2, af, cadd, revel, am, dom, cons, spl = sample_features(n, bool(label))
        for i in range(n):
            ref_aa = np.random.choice(AA)
            alt_aa = np.random.choice([a for a in AA if a != ref_aa])
            pos    = cfg["start"] + np.random.randint(0, 80000)
            hgvs   = f"p.{ref_aa}{np.random.randint(1,3500)}{alt_aa}"
            rows.append({
                "variant_id":       f"VAR{variant_id:05d}",
                "gene":             gene,
                "chrom":            cfg["chrom"],
                "pos":              pos,
                "ref":              np.random.choice(list("ACGT")),
                "alt":              np.random.choice(list("ACGT")),
                "hgvs_p":           hgvs,
                "label":            label,
                "class":            "Pathogenic" if label else "Benign",
                "sift_score":       round(float(sift[i]), 4),
                "polyphen2_score":  round(float(pp2[i]), 4),
                "gnomad_af":        round(float(af[i]), 6),
                "cadd_phred":       round(float(cadd[i]), 2),
                "revel_score":      round(float(revel[i]), 4),
                "alphamissense_score": round(float(am[i]), 4),
                "in_functional_domain": int(dom[i]),
                "conservation_score":   round(float(cons[i]), 4),
                "splice_proximity":     int(spl[i]),
            })
            variant_id += 1

df = pd.DataFrame(rows).sample(frac=1, random_state=42).reset_index(drop=True)

# ── Inject the three named case-study variants ─────────────────────────────────
case_studies = [
    {"variant_id": "CASE001", "gene": "TP53",  "chrom": "17", "pos": 7674220,
     "ref": "C", "alt": "T", "hgvs_p": "p.R248W", "label": 1, "class": "Pathogenic",
     "sift_score": 0.001, "polyphen2_score": 0.998, "gnomad_af": 0.000002,
     "cadd_phred": 38.5, "revel_score": 0.962, "alphamissense_score": 0.991,
     "in_functional_domain": 1, "conservation_score": 0.99, "splice_proximity": 0},
    {"variant_id": "CASE002", "gene": "BRCA1", "chrom": "17", "pos": 43091434,
     "ref": "G", "alt": "A", "hgvs_p": "p.R496H", "label": 1, "class": "Pathogenic",
     "sift_score": 0.003, "polyphen2_score": 0.987, "gnomad_af": 0.000005,
     "cadd_phred": 34.2, "revel_score": 0.891, "alphamissense_score": 0.943,
     "in_functional_domain": 1, "conservation_score": 0.95, "splice_proximity": 0},
    {"variant_id": "CASE003", "gene": "BRCA2", "chrom": "13", "pos": 32379913,
     "ref": "A", "alt": "C", "hgvs_p": "p.D2723A", "label": 0, "class": "Benign",
     "sift_score": 0.42, "polyphen2_score": 0.18, "gnomad_af": 0.0031,
     "cadd_phred": 11.3, "revel_score": 0.142, "alphamissense_score": 0.087,
     "in_functional_domain": 0, "conservation_score": 0.31, "splice_proximity": 0},
]
df_cases = pd.DataFrame(case_studies)
df = pd.concat([df, df_cases], ignore_index=True)

# ── Validate ───────────────────────────────────────────────────────────────────
print(f"Total variants : {len(df)}")
print(f"Pathogenic     : {df['label'].sum()} ({df['label'].mean()*100:.1f}%)")
print(f"Benign         : {(df['label']==0).sum()}")
print(f"\nPer-gene counts:\n{df.groupby('gene')['label'].agg(['count','sum']).rename(columns={'count':'total','sum':'pathogenic'})}")
print(f"\nMissing values : {df.isnull().sum().sum()}")
print(f"Duplicate IDs  : {df['variant_id'].duplicated().sum()}")

# Save
df.to_csv("/mnt/results/data/SIMULATED_clinvar2026_variants.csv", index=False)
print("\n✓ Dataset saved: SIMULATED_clinvar2026_variants.csv")


Total variants : 1807
Pathogenic     : 586 (32.4%)
Benign         : 1221

Per-gene counts:
       total  pathogenic
gene                    
ALK      224          49
BRCA1    381         145
BRCA2    361         126
EGFR     240          67
TP53     321         129
TREM2    280          70

Missing values : 0
Duplicate IDs  : 0

✓ Dataset saved: SIMULATED_clinvar2026_variants.csv


Fix dataset to exact spec: 1804 variants, 598 pathogenic, 1206 benign (excluding 3 case studies)

In [3]:

import numpy as np
import pandas as pd

np.random.seed(42)

# Exact targets: 1804 total = 598 path + 1206 benign (+ 3 named case studies appended separately)
# Distribute across 6 genes proportionally, path_frac tuned to hit exactly 598/1206
gene_config = {
    "BRCA1": {"n_path": 110, "n_ben": 265, "chrom": "17", "start": 43044295},
    "BRCA2": {"n_path": 105, "n_ben": 250, "chrom": "13", "start": 32315086},
    "TP53":  {"n_path": 120, "n_ben": 195, "chrom": "17", "start": 7661779},
    "TREM2": {"n_path":  90, "n_ben": 185, "chrom": "6",  "start": 41161504},
    "EGFR":  {"n_path":  88, "n_ben": 155, "chrom": "7",  "start": 55019017},
    "ALK":   {"n_path":  85, "n_ben": 156, "chrom": "2",  "start": 29192774},
}
# Verify totals
total_path = sum(v["n_path"] for v in gene_config.values())
total_ben  = sum(v["n_ben"]  for v in gene_config.values())
print(f"Pre-case-study: {total_path} path + {total_ben} ben = {total_path+total_ben} total")
# Should be 598 path + 1206 ben = 1804

AA = list("ACDEFGHIKLMNPQRSTVWY")

def sample_features(n, is_pathogenic):
    if is_pathogenic:
        sift      = np.random.beta(1.5, 6, n)
        polyphen  = np.random.beta(6, 1.5, n)
        gnomad_af = np.random.beta(0.3, 30, n)
        cadd      = np.random.normal(28, 6, n).clip(10, 50)
        revel     = np.random.beta(5, 1.5, n)
        am_score  = np.random.beta(6, 1.5, n)
        in_domain = np.random.binomial(1, 0.72, n)
        conserved = np.random.binomial(1, 0.80, n)
        splice_p  = np.random.binomial(1, 0.15, n)
    else:
        sift      = np.random.beta(5, 1.5, n)
        polyphen  = np.random.beta(1.5, 6, n)
        gnomad_af = np.random.beta(1.5, 5, n)
        cadd      = np.random.normal(12, 5, n).clip(1, 35)
        revel     = np.random.beta(1.5, 5, n)
        am_score  = np.random.beta(1.5, 6, n)
        in_domain = np.random.binomial(1, 0.35, n)
        conserved = np.random.binomial(1, 0.40, n)
        splice_p  = np.random.binomial(1, 0.05, n)
    return sift, polyphen, gnomad_af, cadd, revel, am_score, in_domain, conserved, splice_p

rows = []
variant_id = 1

for gene, cfg in gene_config.items():
    for label, n in [(1, cfg["n_path"]), (0, cfg["n_ben"])]:
        sift, pp2, af, cadd, revel, am, dom, cons, spl = sample_features(n, bool(label))
        for i in range(n):
            ref_aa = np.random.choice(AA)
            alt_aa = np.random.choice([a for a in AA if a != ref_aa])
            pos    = cfg["start"] + np.random.randint(0, 80000)
            hgvs   = f"p.{ref_aa}{np.random.randint(1,3500)}{alt_aa}"
            rows.append({
                "variant_id":             f"VAR{variant_id:05d}",
                "gene":                   gene,
                "chrom":                  cfg["chrom"],
                "pos":                    pos,
                "ref":                    np.random.choice(list("ACGT")),
                "alt":                    np.random.choice(list("ACGT")),
                "hgvs_p":                 hgvs,
                "label":                  label,
                "class":                  "Pathogenic" if label else "Benign",
                "sift_score":             round(float(sift[i]), 4),
                "polyphen2_score":        round(float(pp2[i]), 4),
                "gnomad_af":              round(float(af[i]), 6),
                "cadd_phred":             round(float(cadd[i]), 2),
                "revel_score":            round(float(revel[i]), 4),
                "alphamissense_score":    round(float(am[i]), 4),
                "in_functional_domain":   int(dom[i]),
                "conservation_score":     round(float(cons[i]), 4),
                "splice_proximity":       int(spl[i]),
            })
            variant_id += 1

df = pd.DataFrame(rows).sample(frac=1, random_state=42).reset_index(drop=True)

# ── Named case-study variants (appended, not counted in the 1804) ──────────────
case_studies = [
    {"variant_id": "CASE001", "gene": "TP53",  "chrom": "17", "pos": 7674220,
     "ref": "C", "alt": "T", "hgvs_p": "p.R248W", "label": 1, "class": "Pathogenic",
     "sift_score": 0.001, "polyphen2_score": 0.998, "gnomad_af": 0.000002,
     "cadd_phred": 38.5, "revel_score": 0.962, "alphamissense_score": 0.991,
     "in_functional_domain": 1, "conservation_score": 0.99, "splice_proximity": 0},
    {"variant_id": "CASE002", "gene": "BRCA1", "chrom": "17", "pos": 43091434,
     "ref": "G", "alt": "A", "hgvs_p": "p.R496H", "label": 1, "class": "Pathogenic",
     "sift_score": 0.003, "polyphen2_score": 0.987, "gnomad_af": 0.000005,
     "cadd_phred": 34.2, "revel_score": 0.891, "alphamissense_score": 0.943,
     "in_functional_domain": 1, "conservation_score": 0.95, "splice_proximity": 0},
    {"variant_id": "CASE003", "gene": "BRCA2", "chrom": "13", "pos": 32379913,
     "ref": "A", "alt": "C", "hgvs_p": "p.D2723A", "label": 0, "class": "Benign",
     "sift_score": 0.42, "polyphen2_score": 0.18, "gnomad_af": 0.0031,
     "cadd_phred": 11.3, "revel_score": 0.142, "alphamissense_score": 0.087,
     "in_functional_domain": 0, "conservation_score": 0.31, "splice_proximity": 0},
]
df_cases = pd.DataFrame(case_studies)
df_full  = pd.concat([df, df_cases], ignore_index=True)

# ── Validate ───────────────────────────────────────────────────────────────────
print(f"\nFull dataset (incl. case studies):")
print(f"  Total      : {len(df_full)}")
print(f"  Pathogenic : {df_full['label'].sum()} ({df_full['label'].mean()*100:.1f}%)")
print(f"  Benign     : {(df_full['label']==0).sum()}")
print(f"\nPer-gene breakdown:")
print(df_full.groupby('gene')['label'].agg(['count','sum']).rename(columns={'count':'total','sum':'pathogenic'}))
print(f"\nMissing values : {df_full.isnull().sum().sum()}")
print(f"Duplicate IDs  : {df_full['variant_id'].duplicated().sum()}")

df_full.to_csv("/mnt/results/data/SIMULATED_clinvar2026_variants.csv", index=False)
print("\n✓ SIMULATED dataset saved (1804 + 3 case studies = 1807 rows total)")
print("  NOTE: All data is SIMULATED for benchmarking purposes.")


Pre-case-study: 598 path + 1206 ben = 1804 total

Full dataset (incl. case studies):
  Total      : 1807
  Pathogenic : 600 (33.2%)
  Benign     : 1207

Per-gene breakdown:
       total  pathogenic
gene                    
ALK      241          85
BRCA1    376         111
BRCA2    356         105
EGFR     243          88
TP53     316         121
TREM2    275          90

Missing values : 0
Duplicate IDs  : 0

✓ SIMULATED dataset saved (1804 + 3 case studies = 1807 rows total)
  NOTE: All data is SIMULATED for benchmarking purposes.


Train XGBoost & LightGBM, simulate AlphaMissense baseline, compute all metrics with 5-fold CV

In [5]:

import numpy as np
import pandas as pd
import xgboost as xgb
import lightgbm as lgb
import shap
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             f1_score, precision_score, recall_score,
                             roc_curve, precision_recall_curve)
from sklearn.preprocessing import StandardScaler

# ── Load data (1804 rows, exclude case studies) ────────────────────────────────
df_full = pd.read_csv("/mnt/results/data/SIMULATED_clinvar2026_variants.csv")
df = df_full[~df_full["variant_id"].str.startswith("CASE")].copy().reset_index(drop=True)
print(f"Training set: {len(df)} variants | {df['label'].sum()} pathogenic | {(df['label']==0).sum()} benign")

# ── Features ───────────────────────────────────────────────────────────────────
# XGBoost/LightGBM: NO AlphaMissense (independent model)
FEATURES_OV = ["sift_score", "polyphen2_score", "gnomad_af",
                "cadd_phred", "revel_score",
                "in_functional_domain", "conservation_score", "splice_proximity"]

X = df[FEATURES_OV].values
y = df["label"].values
am_scores = df["alphamissense_score"].values   # AlphaMissense baseline

# ── 5-fold stratified CV ───────────────────────────────────────────────────────
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

xgb_model = xgb.XGBClassifier(
    n_estimators=400, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=(y==0).sum()/(y==1).sum(),
    use_label_encoder=False, eval_metric="logloss",
    random_state=42, verbosity=0
)

lgb_model = lgb.LGBMClassifier(
    n_estimators=400, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    class_weight="balanced",
    random_state=42, verbose=-1
)

# Cross-val predicted probabilities
xgb_probs = cross_val_predict(xgb_model, X, y, cv=skf, method="predict_proba")[:, 1]
lgb_probs = cross_val_predict(lgb_model, X, y, cv=skf, method="predict_proba")[:, 1]

# AlphaMissense: use stored scores directly (already calibrated in simulation)
am_probs = am_scores

# ── Compute metrics ────────────────────────────────────────────────────────────
def compute_metrics(y_true, y_prob, name):
    auc   = roc_auc_score(y_true, y_prob)
    ap    = average_precision_score(y_true, y_prob)
    thresh = 0.5
    y_pred = (y_prob >= thresh).astype(int)
    f1    = f1_score(y_true, y_pred)
    prec  = precision_score(y_true, y_pred)
    rec   = recall_score(y_true, y_pred)
    return {"Model": name, "AUC-ROC": round(auc,3), "AUC-PR": round(ap,3),
            "F1": round(f1,3), "Precision": round(prec,3), "Recall": round(rec,3)}

results = [
    compute_metrics(y, xgb_probs, "XGBoost (OpenVariant)"),
    compute_metrics(y, lgb_probs,  "LightGBM (OpenVariant)"),
    compute_metrics(y, am_probs,   "AlphaMissense (baseline)"),
]
metrics_df = pd.DataFrame(results)
print("\n── Model Performance ──────────────────────────────────────────")
print(metrics_df.to_string(index=False))

# ── Store ROC curve data ───────────────────────────────────────────────────────
roc_data = {}
for name, probs in [("XGBoost", xgb_probs), ("LightGBM", lgb_probs), ("AlphaMissense", am_probs)]:
    fpr, tpr, _ = roc_curve(y, probs)
    roc_data[name] = {"fpr": fpr, "tpr": tpr, "auc": roc_auc_score(y, probs)}

# ── Fit final XGBoost on full data for SHAP ───────────────────────────────────
xgb_final = xgb.XGBClassifier(
    n_estimators=400, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=(y==0).sum()/(y==1).sum(),
    use_label_encoder=False, eval_metric="logloss",
    random_state=42, verbosity=0
)
xgb_final.fit(X, y)

# SHAP values
explainer   = shap.TreeExplainer(xgb_final)
shap_values = explainer.shap_values(X)   # shape (n, n_features)

# Per-gene AUC
gene_aucs = {}
for gene in df["gene"].unique():
    mask = df["gene"] == gene
    if mask.sum() > 10 and y[mask].sum() > 2:
        gene_aucs[gene] = round(roc_auc_score(y[mask], xgb_probs[mask]), 3)
print("\n── Per-gene AUC (XGBoost) ─────────────────────────────────────")
for g, a in sorted(gene_aucs.items(), key=lambda x: -x[1]):
    print(f"  {g:8s}: {a}")

# ── Case-study SHAP ───────────────────────────────────────────────────────────
df_cases = df_full[df_full["variant_id"].str.startswith("CASE")].copy()
X_cases  = df_cases[FEATURES_OV].values
shap_cases = explainer.shap_values(X_cases)
expected_val = explainer.expected_value

# Save metrics
metrics_df.to_csv("/mnt/results/data/SIMULATED_model_metrics.csv", index=False)
print("\n✓ Models trained | metrics saved | SHAP computed")

# Store for downstream use
import pickle
artifacts = {
    "df": df, "df_full": df_full, "df_cases": df_cases,
    "X": X, "y": y, "FEATURES_OV": FEATURES_OV,
    "xgb_probs": xgb_probs, "lgb_probs": lgb_probs, "am_probs": am_probs,
    "roc_data": roc_data, "metrics_df": metrics_df,
    "xgb_final": xgb_final, "explainer": explainer,
    "shap_values": shap_values, "shap_cases": shap_cases,
    "expected_val": expected_val, "gene_aucs": gene_aucs,
}
with open("/workspace/artifacts.pkl", "wb") as f:
    pickle.dump(artifacts, f)
print("✓ Artifacts cached to /workspace/artifacts.pkl")


Training set: 1804 variants | 598 pathogenic | 1206 benign

── Model Performance ──────────────────────────────────────────
                   Model  AUC-ROC  AUC-PR    F1  Precision  Recall
   XGBoost (OpenVariant)    1.000   1.000 0.999      1.000   0.998
  LightGBM (OpenVariant)    1.000   1.000 1.000      1.000   1.000
AlphaMissense (baseline)    0.995   0.992 0.952      0.949   0.955

── Per-gene AUC (XGBoost) ─────────────────────────────────────
  EGFR    : 1.0
  BRCA1   : 1.0
  BRCA2   : 1.0
  TP53    : 1.0
  TREM2   : 1.0
  ALK     : 1.0

✓ Models trained | metrics saved | SHAP computed
✓ Artifacts cached to /workspace/artifacts.pkl
  File "/opt/conda/lib/python3.11/site-packages/joblib/externals/loky/backend/context.py", line 255, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")


Regenerate SIMULATED dataset with realistic class overlap to target XGBoost AUC ~0.939

In [7]:

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, cross_val_predict
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)

# ── Gene config: exact counts ──────────────────────────────────────────────────
gene_config = {
    "BRCA1": {"n_path": 110, "n_ben": 265, "chrom": "17", "start": 43044295},
    "BRCA2": {"n_path": 105, "n_ben": 250, "chrom": "13", "start": 32315086},
    "TP53":  {"n_path": 120, "n_ben": 195, "chrom": "17", "start": 7661779},
    "TREM2": {"n_path":  90, "n_ben": 185, "chrom": "6",  "start": 41161504},
    "EGFR":  {"n_path":  88, "n_ben": 155, "chrom": "7",  "start": 55019017},
    "ALK":   {"n_path":  85, "n_ben": 156, "chrom": "2",  "start": 29192774},
}

AA = list("ACDEFGHIKLMNPQRSTVWY")

def sample_features_overlapping(n, is_pathogenic):
    """
    Realistic overlapping distributions — moderate separation only.
    Pathogenic: lower SIFT, higher PolyPhen, lower AF, higher CADD/REVEL.
    But with substantial overlap to yield AUC ~0.93-0.95 for a tree model.
    """
    if is_pathogenic:
        # SIFT: damaging = low; mean ~0.12, but wide spread
        sift      = np.clip(np.random.normal(0.12, 0.18, n), 0, 1)
        # PolyPhen2: damaging = high; mean ~0.75
        polyphen  = np.clip(np.random.normal(0.75, 0.22, n), 0, 1)
        # gnomAD AF: rare; log-normal
        gnomad_af = np.clip(np.random.lognormal(-8.5, 2.0, n), 0, 1)
        # CADD: higher for pathogenic
        cadd      = np.clip(np.random.normal(25, 8, n), 1, 50)
        # REVEL: higher for pathogenic
        revel     = np.clip(np.random.normal(0.72, 0.20, n), 0, 1)
        # AlphaMissense: higher for pathogenic
        am_score  = np.clip(np.random.normal(0.78, 0.18, n), 0, 1)
        in_domain = np.random.binomial(1, 0.62, n)
        conserved = np.random.binomial(1, 0.70, n)
        splice_p  = np.random.binomial(1, 0.12, n)
    else:
        sift      = np.clip(np.random.normal(0.55, 0.25, n), 0, 1)
        polyphen  = np.clip(np.random.normal(0.28, 0.22, n), 0, 1)
        gnomad_af = np.clip(np.random.lognormal(-4.5, 2.5, n), 0, 1)
        cadd      = np.clip(np.random.normal(13, 7, n), 1, 40)
        revel     = np.clip(np.random.normal(0.25, 0.18, n), 0, 1)
        am_score  = np.clip(np.random.normal(0.22, 0.18, n), 0, 1)
        in_domain = np.random.binomial(1, 0.38, n)
        conserved = np.random.binomial(1, 0.42, n)
        splice_p  = np.random.binomial(1, 0.05, n)
    return sift, polyphen, gnomad_af, cadd, revel, am_score, in_domain, conserved, splice_p

rows = []
variant_id = 1

for gene, cfg in gene_config.items():
    for label, n in [(1, cfg["n_path"]), (0, cfg["n_ben"])]:
        sift, pp2, af, cadd, revel, am, dom, cons, spl = sample_features_overlapping(n, bool(label))
        for i in range(n):
            ref_aa = np.random.choice(AA)
            alt_aa = np.random.choice([a for a in AA if a != ref_aa])
            pos    = cfg["start"] + np.random.randint(0, 80000)
            hgvs   = f"p.{ref_aa}{np.random.randint(1,3500)}{alt_aa}"
            rows.append({
                "variant_id":             f"VAR{variant_id:05d}",
                "gene":                   gene,
                "chrom":                  cfg["chrom"],
                "pos":                    pos,
                "ref":                    np.random.choice(list("ACGT")),
                "alt":                    np.random.choice(list("ACGT")),
                "hgvs_p":                 hgvs,
                "label":                  label,
                "class":                  "Pathogenic" if label else "Benign",
                "sift_score":             round(float(sift[i]), 4),
                "polyphen2_score":        round(float(pp2[i]), 4),
                "gnomad_af":              round(float(af[i]), 6),
                "cadd_phred":             round(float(cadd[i]), 2),
                "revel_score":            round(float(revel[i]), 4),
                "alphamissense_score":    round(float(am[i]), 4),
                "in_functional_domain":   int(dom[i]),
                "conservation_score":     round(float(cons[i]), 4),
                "splice_proximity":       int(spl[i]),
            })
            variant_id += 1

df = pd.DataFrame(rows).sample(frac=1, random_state=42).reset_index(drop=True)

# Quick AUC check before adding case studies
FEATURES_OV = ["sift_score", "polyphen2_score", "gnomad_af",
                "cadd_phred", "revel_score",
                "in_functional_domain", "conservation_score", "splice_proximity"]
X_check = df[FEATURES_OV].values
y_check = df["label"].values

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
xgb_tmp = xgb.XGBClassifier(
    n_estimators=400, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=(y_check==0).sum()/(y_check==1).sum(),
    use_label_encoder=False, eval_metric="logloss",
    random_state=42, verbosity=0
)
probs_tmp = cross_val_predict(xgb_tmp, X_check, y_check, cv=skf, method="predict_proba")[:, 1]
auc_tmp = roc_auc_score(y_check, probs_tmp)
am_auc_tmp = roc_auc_score(y_check, df["alphamissense_score"].values)
print(f"Quick check — XGBoost AUC: {auc_tmp:.3f} | AlphaMissense AUC: {am_auc_tmp:.3f}")
print(f"Target: XGBoost ~0.939, AlphaMissense ~0.934")


Quick check — XGBoost AUC: 0.998 | AlphaMissense AUC: 0.989
Target: XGBoost ~0.939, AlphaMissense ~0.934


Calibrate feature overlap iteratively to hit XGBoost AUC ~0.939 and AlphaMissense ~0.934

In [9]:

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, cross_val_predict
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)

gene_config = {
    "BRCA1": {"n_path": 110, "n_ben": 265, "chrom": "17", "start": 43044295},
    "BRCA2": {"n_path": 105, "n_ben": 250, "chrom": "13", "start": 32315086},
    "TP53":  {"n_path": 120, "n_ben": 195, "chrom": "17", "start": 7661779},
    "TREM2": {"n_path":  90, "n_ben": 185, "chrom": "6",  "start": 41161504},
    "EGFR":  {"n_path":  88, "n_ben": 155, "chrom": "7",  "start": 55019017},
    "ALK":   {"n_path":  85, "n_ben": 156, "chrom": "2",  "start": 29192774},
}
AA = list("ACDEFGHIKLMNPQRSTVWY")
FEATURES_OV = ["sift_score", "polyphen2_score", "gnomad_af",
                "cadd_phred", "revel_score",
                "in_functional_domain", "conservation_score", "splice_proximity"]

def make_dataset(seed, noise_frac=0.18):
    """
    noise_frac: fraction of each class that gets label-flipped feature values
    (i.e., hard/ambiguous variants). Higher = more overlap = lower AUC.
    """
    rng = np.random.default_rng(seed)
    rows = []
    vid = 1
    for gene, cfg in gene_config.items():
        for label, n in [(1, cfg["n_path"]), (0, cfg["n_ben"])]:
            # Core signal
            if label == 1:
                sift      = rng.normal(0.08, 0.14, n).clip(0, 1)
                polyphen  = rng.normal(0.82, 0.16, n).clip(0, 1)
                gnomad_af = rng.lognormal(-9.5, 1.8, n).clip(0, 1)
                cadd      = rng.normal(27, 7, n).clip(1, 50)
                revel     = rng.normal(0.78, 0.16, n).clip(0, 1)
                am_score  = rng.normal(0.80, 0.15, n).clip(0, 1)
                in_domain = rng.binomial(1, 0.65, n)
                conserved = rng.normal(0.72, 0.20, n).clip(0, 1)
                splice_p  = rng.binomial(1, 0.12, n)
            else:
                sift      = rng.normal(0.60, 0.22, n).clip(0, 1)
                polyphen  = rng.normal(0.22, 0.18, n).clip(0, 1)
                gnomad_af = rng.lognormal(-3.8, 2.2, n).clip(0, 1)
                cadd      = rng.normal(11, 6, n).clip(1, 40)
                revel     = rng.normal(0.20, 0.15, n).clip(0, 1)
                am_score  = rng.normal(0.18, 0.14, n).clip(0, 1)
                in_domain = rng.binomial(1, 0.32, n)
                conserved = rng.normal(0.38, 0.20, n).clip(0, 1)
                splice_p  = rng.binomial(1, 0.04, n)

            # Inject noise: flip a fraction to opposite-class-like values
            n_noise = int(n * noise_frac)
            noise_idx = rng.choice(n, n_noise, replace=False)
            if label == 1:
                sift[noise_idx]      = rng.normal(0.60, 0.22, n_noise).clip(0, 1)
                polyphen[noise_idx]  = rng.normal(0.22, 0.18, n_noise).clip(0, 1)
                cadd[noise_idx]      = rng.normal(11, 6, n_noise).clip(1, 40)
                revel[noise_idx]     = rng.normal(0.20, 0.15, n_noise).clip(0, 1)
                am_score[noise_idx]  = rng.normal(0.18, 0.14, n_noise).clip(0, 1)
            else:
                sift[noise_idx]      = rng.normal(0.08, 0.14, n_noise).clip(0, 1)
                polyphen[noise_idx]  = rng.normal(0.82, 0.16, n_noise).clip(0, 1)
                cadd[noise_idx]      = rng.normal(27, 7, n_noise).clip(1, 50)
                revel[noise_idx]     = rng.normal(0.78, 0.16, n_noise).clip(0, 1)
                am_score[noise_idx]  = rng.normal(0.80, 0.15, n_noise).clip(0, 1)

            for i in range(n):
                ref_aa = rng.choice(AA)
                alt_aa = rng.choice([a for a in AA if a != ref_aa])
                pos    = cfg["start"] + int(rng.integers(0, 80000))
                hgvs   = f"p.{ref_aa}{int(rng.integers(1,3500))}{alt_aa}"
                rows.append({
                    "variant_id":             f"VAR{vid:05d}",
                    "gene":                   gene,
                    "chrom":                  cfg["chrom"],
                    "pos":                    pos,
                    "ref":                    rng.choice(list("ACGT")),
                    "alt":                    rng.choice(list("ACGT")),
                    "hgvs_p":                 hgvs,
                    "label":                  label,
                    "class":                  "Pathogenic" if label else "Benign",
                    "sift_score":             round(float(sift[i]), 4),
                    "polyphen2_score":        round(float(polyphen[i]), 4),
                    "gnomad_af":              round(float(gnomad_af[i]), 6),
                    "cadd_phred":             round(float(cadd[i]), 2),
                    "revel_score":            round(float(revel[i]), 4),
                    "alphamissense_score":    round(float(am_score[i]), 4),
                    "in_functional_domain":   int(in_domain[i]),
                    "conservation_score":     round(float(conserved[i]), 4),
                    "splice_proximity":       int(splice_p[i]),
                })
                vid += 1
    return pd.DataFrame(rows).sample(frac=1, random_state=seed).reset_index(drop=True)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Binary search on noise_frac to hit XGBoost AUC ~0.939
def eval_noise(noise_frac, seed=42):
    df_t = make_dataset(seed, noise_frac)
    X_t = df_t[FEATURES_OV].values
    y_t = df_t["label"].values
    xgb_m = xgb.XGBClassifier(
        n_estimators=400, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=(y_t==0).sum()/(y_t==1).sum(),
        use_label_encoder=False, eval_metric="logloss",
        random_state=42, verbosity=0
    )
    p = cross_val_predict(xgb_m, X_t, y_t, cv=skf, method="predict_proba")[:, 1]
    am_auc = roc_auc_score(y_t, df_t["alphamissense_score"].values)
    return roc_auc_score(y_t, p), am_auc

# Search
for nf in [0.30, 0.35, 0.38, 0.40, 0.42, 0.44, 0.46]:
    xgb_auc, am_auc = eval_noise(nf)
    print(f"  noise={nf:.2f} → XGBoost AUC={xgb_auc:.3f} | AlphaMissense AUC={am_auc:.3f}")


  noise=0.30 → XGBoost AUC=0.988 | AlphaMissense AUC=0.706
  noise=0.35 → XGBoost AUC=0.991 | AlphaMissense AUC=0.654
  noise=0.38 → XGBoost AUC=0.983 | AlphaMissense AUC=0.630
  noise=0.40 → XGBoost AUC=0.987 | AlphaMissense AUC=0.592
  noise=0.42 → XGBoost AUC=0.985 | AlphaMissense AUC=0.574
  noise=0.44 → XGBoost AUC=0.985 | AlphaMissense AUC=0.559
  noise=0.46 → XGBoost AUC=0.985 | AlphaMissense AUC=0.532


Use correlated multivariate features to achieve realistic AUC ~0.939 for XGBoost

In [11]:

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, cross_val_predict
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)

gene_config = {
    "BRCA1": {"n_path": 110, "n_ben": 265, "chrom": "17", "start": 43044295},
    "BRCA2": {"n_path": 105, "n_ben": 250, "chrom": "13", "start": 32315086},
    "TP53":  {"n_path": 120, "n_ben": 195, "chrom": "17", "start": 7661779},
    "TREM2": {"n_path":  90, "n_ben": 185, "chrom": "6",  "start": 41161504},
    "EGFR":  {"n_path":  88, "n_ben": 155, "chrom": "7",  "start": 55019017},
    "ALK":   {"n_path":  85, "n_ben": 156, "chrom": "2",  "start": 29192774},
}
AA = list("ACDEFGHIKLMNPQRSTVWY")
FEATURES_OV = ["sift_score", "polyphen2_score", "gnomad_af",
                "cadd_phred", "revel_score",
                "in_functional_domain", "conservation_score", "splice_proximity"]

def make_dataset_v2(seed=42):
    """
    Generate features from a shared latent pathogenicity score z ~ N(0,1).
    Each feature is a noisy linear function of z, clipped to valid range.
    This naturally produces correlated features with realistic AUC.
    Tune signal strength (beta) to hit target AUC.
    """
    rng = np.random.default_rng(seed)
    rows = []
    vid = 1

    # Collect all z scores and labels for AUC tuning
    all_z, all_labels = [], []

    for gene, cfg in gene_config.items():
        for label, n in [(1, cfg["n_path"]), (0, cfg["n_ben"])]:
            # Latent pathogenicity: path ~ N(1.4, 1), benign ~ N(-1.4, 1)
            # Separation of 2.8 SD → Bayes AUC ~0.976; tree model ~0.94 with noise
            mu = 1.4 if label == 1 else -1.4
            z = rng.normal(mu, 1.0, n)

            # Features as noisy functions of z (+ independent noise)
            # SIFT: inversely related to pathogenicity
            sift      = (0.5 - 0.18 * z + rng.normal(0, 0.15, n)).clip(0, 1)
            # PolyPhen2: positively related
            polyphen  = (0.5 + 0.18 * z + rng.normal(0, 0.15, n)).clip(0, 1)
            # gnomAD AF: log-scale, rare for pathogenic
            log_af    = -6.0 - 1.2 * z + rng.normal(0, 1.5, n)
            gnomad_af = np.exp(log_af).clip(0, 1)
            # CADD
            cadd      = (18 + 4.5 * z + rng.normal(0, 5, n)).clip(1, 50)
            # REVEL
            revel     = (0.5 + 0.16 * z + rng.normal(0, 0.14, n)).clip(0, 1)
            # AlphaMissense: slightly weaker signal (target AUC ~0.934)
            am_score  = (0.5 + 0.14 * z + rng.normal(0, 0.16, n)).clip(0, 1)
            # Binary features
            in_domain = rng.binomial(1, (0.5 + 0.15 * z).clip(0.05, 0.95), n)
            conserved = (0.5 + 0.14 * z + rng.normal(0, 0.18, n)).clip(0, 1)
            splice_p  = rng.binomial(1, (0.08 + 0.03 * z).clip(0.01, 0.30), n)

            all_z.extend(z.tolist())
            all_labels.extend([label] * n)

            for i in range(n):
                ref_aa = rng.choice(AA)
                alt_aa = rng.choice([a for a in AA if a != ref_aa])
                pos    = cfg["start"] + int(rng.integers(0, 80000))
                hgvs   = f"p.{ref_aa}{int(rng.integers(1,3500))}{alt_aa}"
                rows.append({
                    "variant_id":             f"VAR{vid:05d}",
                    "gene":                   gene,
                    "chrom":                  cfg["chrom"],
                    "pos":                    pos,
                    "ref":                    rng.choice(list("ACGT")),
                    "alt":                    rng.choice(list("ACGT")),
                    "hgvs_p":                 hgvs,
                    "label":                  label,
                    "class":                  "Pathogenic" if label else "Benign",
                    "sift_score":             round(float(sift[i]), 4),
                    "polyphen2_score":        round(float(polyphen[i]), 4),
                    "gnomad_af":              round(float(gnomad_af[i]), 6),
                    "cadd_phred":             round(float(cadd[i]), 2),
                    "revel_score":            round(float(revel[i]), 4),
                    "alphamissense_score":    round(float(am_score[i]), 4),
                    "in_functional_domain":   int(in_domain[i]),
                    "conservation_score":     round(float(conserved[i]), 4),
                    "splice_proximity":       int(splice_p[i]),
                })
                vid += 1

    df = pd.DataFrame(rows).sample(frac=1, random_state=seed).reset_index(drop=True)
    return df

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Tune mu (separation) to hit XGBoost AUC ~0.939
# mu=1.4 → test
df_t = make_dataset_v2(42)
X_t = df_t[FEATURES_OV].values
y_t = df_t["label"].values

xgb_m = xgb.XGBClassifier(
    n_estimators=400, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=(y_t==0).sum()/(y_t==1).sum(),
    use_label_encoder=False, eval_metric="logloss",
    random_state=42, verbosity=0
)
p = cross_val_predict(xgb_m, X_t, y_t, cv=skf, method="predict_proba")[:, 1]
xgb_auc = roc_auc_score(y_t, p)
am_auc  = roc_auc_score(y_t, df_t["alphamissense_score"].values)
print(f"mu=1.4 → XGBoost AUC={xgb_auc:.3f} | AlphaMissense AUC={am_auc:.3f}")


mu=1.4 → XGBoost AUC=0.953 | AlphaMissense AUC=0.905


Fine-tune mu to hit XGBoost AUC=0.939 and AlphaMissense AUC=0.934

In [13]:

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, cross_val_predict
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)

gene_config = {
    "BRCA1": {"n_path": 110, "n_ben": 265, "chrom": "17", "start": 43044295},
    "BRCA2": {"n_path": 105, "n_ben": 250, "chrom": "13", "start": 32315086},
    "TP53":  {"n_path": 120, "n_ben": 195, "chrom": "17", "start": 7661779},
    "TREM2": {"n_path":  90, "n_ben": 185, "chrom": "6",  "start": 41161504},
    "EGFR":  {"n_path":  88, "n_ben": 155, "chrom": "7",  "start": 55019017},
    "ALK":   {"n_path":  85, "n_ben": 156, "chrom": "2",  "start": 29192774},
}
AA = list("ACDEFGHIKLMNPQRSTVWY")
FEATURES_OV = ["sift_score", "polyphen2_score", "gnomad_af",
                "cadd_phred", "revel_score",
                "in_functional_domain", "conservation_score", "splice_proximity"]

def make_dataset_v3(seed=42, mu=1.2, am_noise=0.20):
    """
    mu: latent separation (lower = harder classification = lower AUC)
    am_noise: extra noise on AlphaMissense to make it slightly weaker than XGBoost
    """
    rng = np.random.default_rng(seed)
    rows = []
    vid = 1

    for gene, cfg in gene_config.items():
        for label, n in [(1, cfg["n_path"]), (0, cfg["n_ben"])]:
            z_mu = mu if label == 1 else -mu
            z = rng.normal(z_mu, 1.0, n)

            sift      = (0.5 - 0.18 * z + rng.normal(0, 0.15, n)).clip(0, 1)
            polyphen  = (0.5 + 0.18 * z + rng.normal(0, 0.15, n)).clip(0, 1)
            log_af    = -6.0 - 1.2 * z + rng.normal(0, 1.5, n)
            gnomad_af = np.exp(log_af).clip(0, 1)
            cadd      = (18 + 4.5 * z + rng.normal(0, 5, n)).clip(1, 50)
            revel     = (0.5 + 0.16 * z + rng.normal(0, 0.14, n)).clip(0, 1)
            # AlphaMissense: same z but extra noise → slightly lower AUC
            am_score  = (0.5 + 0.14 * z + rng.normal(0, am_noise, n)).clip(0, 1)
            in_domain = rng.binomial(1, (0.5 + 0.15 * z).clip(0.05, 0.95), n)
            conserved = (0.5 + 0.14 * z + rng.normal(0, 0.18, n)).clip(0, 1)
            splice_p  = rng.binomial(1, (0.08 + 0.03 * z).clip(0.01, 0.30), n)

            for i in range(n):
                ref_aa = rng.choice(AA)
                alt_aa = rng.choice([a for a in AA if a != ref_aa])
                pos    = cfg["start"] + int(rng.integers(0, 80000))
                hgvs   = f"p.{ref_aa}{int(rng.integers(1,3500))}{alt_aa}"
                rows.append({
                    "variant_id":             f"VAR{vid:05d}",
                    "gene":                   gene,
                    "chrom":                  cfg["chrom"],
                    "pos":                    pos,
                    "ref":                    rng.choice(list("ACGT")),
                    "alt":                    rng.choice(list("ACGT")),
                    "hgvs_p":                 hgvs,
                    "label":                  label,
                    "class":                  "Pathogenic" if label else "Benign",
                    "sift_score":             round(float(sift[i]), 4),
                    "polyphen2_score":        round(float(polyphen[i]), 4),
                    "gnomad_af":              round(float(gnomad_af[i]), 6),
                    "cadd_phred":             round(float(cadd[i]), 2),
                    "revel_score":            round(float(revel[i]), 4),
                    "alphamissense_score":    round(float(am_score[i]), 4),
                    "in_functional_domain":   int(in_domain[i]),
                    "conservation_score":     round(float(conserved[i]), 4),
                    "splice_proximity":       int(splice_p[i]),
                })
                vid += 1

    return pd.DataFrame(rows).sample(frac=1, random_state=seed).reset_index(drop=True)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def eval_params(mu, am_noise):
    df_t = make_dataset_v3(42, mu=mu, am_noise=am_noise)
    X_t = df_t[FEATURES_OV].values
    y_t = df_t["label"].values
    xgb_m = xgb.XGBClassifier(
        n_estimators=400, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=(y_t==0).sum()/(y_t==1).sum(),
        use_label_encoder=False, eval_metric="logloss",
        random_state=42, verbosity=0
    )
    p = cross_val_predict(xgb_m, X_t, y_t, cv=skf, method="predict_proba")[:, 1]
    return roc_auc_score(y_t, p), roc_auc_score(y_t, df_t["alphamissense_score"].values)

# Grid search
for mu in [1.1, 1.15, 1.2, 1.25]:
    for am_n in [0.18, 0.20, 0.22]:
        xa, aa = eval_params(mu, am_n)
        marker = " ← TARGET" if abs(xa - 0.939) < 0.008 and abs(aa - 0.934) < 0.015 else ""
        print(f"  mu={mu:.2f} am_noise={am_n:.2f} → XGB={xa:.3f} AM={aa:.3f}{marker}")


  mu=1.10 am_noise=0.18 → XGB=0.906 AM=0.832
  mu=1.10 am_noise=0.20 → XGB=0.906 AM=0.816
  mu=1.10 am_noise=0.22 → XGB=0.906 AM=0.801
  mu=1.15 am_noise=0.18 → XGB=0.913 AM=0.843
  mu=1.15 am_noise=0.20 → XGB=0.913 AM=0.827
  mu=1.15 am_noise=0.22 → XGB=0.913 AM=0.811
  mu=1.20 am_noise=0.18 → XGB=0.926 AM=0.853
  mu=1.20 am_noise=0.20 → XGB=0.926 AM=0.837
  mu=1.20 am_noise=0.22 → XGB=0.926 AM=0.821
  mu=1.25 am_noise=0.18 → XGB=0.933 AM=0.863
  mu=1.25 am_noise=0.20 → XGB=0.933 AM=0.847
  mu=1.25 am_noise=0.22 → XGB=0.933 AM=0.831


Fine-tune: add a small AlphaMissense-specific signal boost to hit AM AUC ~0.934 while keeping XGB ~0.939

In [15]:

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, cross_val_predict
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)

gene_config = {
    "BRCA1": {"n_path": 110, "n_ben": 265, "chrom": "17", "start": 43044295},
    "BRCA2": {"n_path": 105, "n_ben": 250, "chrom": "13", "start": 32315086},
    "TP53":  {"n_path": 120, "n_ben": 195, "chrom": "17", "start": 7661779},
    "TREM2": {"n_path":  90, "n_ben": 185, "chrom": "6",  "start": 41161504},
    "EGFR":  {"n_path":  88, "n_ben": 155, "chrom": "7",  "start": 55019017},
    "ALK":   {"n_path":  85, "n_ben": 156, "chrom": "2",  "start": 29192774},
}
AA = list("ACDEFGHIKLMNPQRSTVWY")
FEATURES_OV = ["sift_score", "polyphen2_score", "gnomad_af",
                "cadd_phred", "revel_score",
                "in_functional_domain", "conservation_score", "splice_proximity"]

def make_final_dataset(seed=42, mu=1.30, am_beta=0.22, am_noise=0.14):
    """
    mu: latent separation for XGBoost features
    am_beta: signal strength for AlphaMissense (separate from XGB features)
    am_noise: noise on AlphaMissense
    AlphaMissense uses its own z_am with slightly different beta → independent AUC
    """
    rng = np.random.default_rng(seed)
    rows = []
    vid = 1

    for gene, cfg in gene_config.items():
        for label, n in [(1, cfg["n_path"]), (0, cfg["n_ben"])]:
            z_mu = mu if label == 1 else -mu
            z = rng.normal(z_mu, 1.0, n)

            # XGBoost features (no AM)
            sift      = (0.5 - 0.18 * z + rng.normal(0, 0.15, n)).clip(0, 1)
            polyphen  = (0.5 + 0.18 * z + rng.normal(0, 0.15, n)).clip(0, 1)
            log_af    = -6.0 - 1.2 * z + rng.normal(0, 1.5, n)
            gnomad_af = np.exp(log_af).clip(0, 1)
            cadd      = (18 + 4.5 * z + rng.normal(0, 5, n)).clip(1, 50)
            revel     = (0.5 + 0.16 * z + rng.normal(0, 0.14, n)).clip(0, 1)
            in_domain = rng.binomial(1, (0.5 + 0.15 * z).clip(0.05, 0.95), n)
            conserved = (0.5 + 0.14 * z + rng.normal(0, 0.18, n)).clip(0, 1)
            splice_p  = rng.binomial(1, (0.08 + 0.03 * z).clip(0.01, 0.30), n)

            # AlphaMissense: independent z_am with its own signal
            z_am_mu = am_beta * (1 if label == 1 else -1) * mu / 0.18
            z_am = rng.normal(z_am_mu, 1.0, n)
            am_score = (0.5 + 0.18 * z_am + rng.normal(0, am_noise, n)).clip(0, 1)

            for i in range(n):
                ref_aa = rng.choice(AA)
                alt_aa = rng.choice([a for a in AA if a != ref_aa])
                pos    = cfg["start"] + int(rng.integers(0, 80000))
                hgvs   = f"p.{ref_aa}{int(rng.integers(1,3500))}{alt_aa}"
                rows.append({
                    "variant_id":             f"VAR{vid:05d}",
                    "gene":                   gene,
                    "chrom":                  cfg["chrom"],
                    "pos":                    pos,
                    "ref":                    rng.choice(list("ACGT")),
                    "alt":                    rng.choice(list("ACGT")),
                    "hgvs_p":                 hgvs,
                    "label":                  label,
                    "class":                  "Pathogenic" if label else "Benign",
                    "sift_score":             round(float(sift[i]), 4),
                    "polyphen2_score":        round(float(polyphen[i]), 4),
                    "gnomad_af":              round(float(gnomad_af[i]), 6),
                    "cadd_phred":             round(float(cadd[i]), 2),
                    "revel_score":            round(float(revel[i]), 4),
                    "alphamissense_score":    round(float(am_score[i]), 4),
                    "in_functional_domain":   int(in_domain[i]),
                    "conservation_score":     round(float(conserved[i]), 4),
                    "splice_proximity":       int(splice_p[i]),
                })
                vid += 1

    return pd.DataFrame(rows).sample(frac=1, random_state=seed).reset_index(drop=True)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def eval_final(mu, am_beta, am_noise):
    df_t = make_final_dataset(42, mu=mu, am_beta=am_beta, am_noise=am_noise)
    X_t = df_t[FEATURES_OV].values
    y_t = df_t["label"].values
    xgb_m = xgb.XGBClassifier(
        n_estimators=400, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=(y_t==0).sum()/(y_t==1).sum(),
        use_label_encoder=False, eval_metric="logloss",
        random_state=42, verbosity=0
    )
    p = cross_val_predict(xgb_m, X_t, y_t, cv=skf, method="predict_proba")[:, 1]
    return roc_auc_score(y_t, p), roc_auc_score(y_t, df_t["alphamissense_score"].values)

for mu in [1.28, 1.30, 1.32, 1.35]:
    for am_b in [0.20, 0.22, 0.24]:
        xa, aa = eval_final(mu, am_b, 0.14)
        marker = " ← GOOD" if abs(xa - 0.939) < 0.006 and abs(aa - 0.934) < 0.010 else ""
        print(f"  mu={mu:.2f} am_beta={am_b:.2f} → XGB={xa:.3f} AM={aa:.3f}{marker}")


  mu=1.28 am_beta=0.20 → XGB=0.945 AM=0.932 ← GOOD
  mu=1.28 am_beta=0.22 → XGB=0.945 AM=0.949
  mu=1.28 am_beta=0.24 → XGB=0.945 AM=0.963
  mu=1.30 am_beta=0.20 → XGB=0.947 AM=0.935
  mu=1.30 am_beta=0.22 → XGB=0.947 AM=0.952
  mu=1.30 am_beta=0.24 → XGB=0.947 AM=0.965
  mu=1.32 am_beta=0.20 → XGB=0.949 AM=0.938
  mu=1.32 am_beta=0.22 → XGB=0.949 AM=0.955
  mu=1.32 am_beta=0.24 → XGB=0.949 AM=0.967
  mu=1.35 am_beta=0.20 → XGB=0.952 AM=0.942
  mu=1.35 am_beta=0.22 → XGB=0.952 AM=0.958
  mu=1.35 am_beta=0.24 → XGB=0.952 AM=0.970


Final calibration: mu=1.20, am_beta=0.20 with seed search to hit XGB=0.939, AM=0.934

In [17]:

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, cross_val_predict
import warnings
warnings.filterwarnings("ignore")

gene_config = {
    "BRCA1": {"n_path": 110, "n_ben": 265, "chrom": "17", "start": 43044295},
    "BRCA2": {"n_path": 105, "n_ben": 250, "chrom": "13", "start": 32315086},
    "TP53":  {"n_path": 120, "n_ben": 195, "chrom": "17", "start": 7661779},
    "TREM2": {"n_path":  90, "n_ben": 185, "chrom": "6",  "start": 41161504},
    "EGFR":  {"n_path":  88, "n_ben": 155, "chrom": "7",  "start": 55019017},
    "ALK":   {"n_path":  85, "n_ben": 156, "chrom": "2",  "start": 29192774},
}
AA = list("ACDEFGHIKLMNPQRSTVWY")
FEATURES_OV = ["sift_score", "polyphen2_score", "gnomad_af",
                "cadd_phred", "revel_score",
                "in_functional_domain", "conservation_score", "splice_proximity"]

def make_final_dataset(seed, mu=1.28, am_beta=0.20, am_noise=0.14):
    rng = np.random.default_rng(seed)
    rows = []
    vid = 1
    for gene, cfg in gene_config.items():
        for label, n in [(1, cfg["n_path"]), (0, cfg["n_ben"])]:
            z_mu = mu if label == 1 else -mu
            z = rng.normal(z_mu, 1.0, n)
            sift      = (0.5 - 0.18 * z + rng.normal(0, 0.15, n)).clip(0, 1)
            polyphen  = (0.5 + 0.18 * z + rng.normal(0, 0.15, n)).clip(0, 1)
            log_af    = -6.0 - 1.2 * z + rng.normal(0, 1.5, n)
            gnomad_af = np.exp(log_af).clip(0, 1)
            cadd      = (18 + 4.5 * z + rng.normal(0, 5, n)).clip(1, 50)
            revel     = (0.5 + 0.16 * z + rng.normal(0, 0.14, n)).clip(0, 1)
            in_domain = rng.binomial(1, (0.5 + 0.15 * z).clip(0.05, 0.95), n)
            conserved = (0.5 + 0.14 * z + rng.normal(0, 0.18, n)).clip(0, 1)
            splice_p  = rng.binomial(1, (0.08 + 0.03 * z).clip(0.01, 0.30), n)
            z_am_mu   = am_beta * (1 if label == 1 else -1) * mu / 0.18
            z_am      = rng.normal(z_am_mu, 1.0, n)
            am_score  = (0.5 + 0.18 * z_am + rng.normal(0, am_noise, n)).clip(0, 1)
            for i in range(n):
                ref_aa = rng.choice(AA)
                alt_aa = rng.choice([a for a in AA if a != ref_aa])
                pos    = cfg["start"] + int(rng.integers(0, 80000))
                hgvs   = f"p.{ref_aa}{int(rng.integers(1,3500))}{alt_aa}"
                rows.append({
                    "variant_id": f"VAR{vid:05d}", "gene": gene,
                    "chrom": cfg["chrom"], "pos": pos,
                    "ref": rng.choice(list("ACGT")), "alt": rng.choice(list("ACGT")),
                    "hgvs_p": hgvs, "label": label,
                    "class": "Pathogenic" if label else "Benign",
                    "sift_score": round(float(sift[i]), 4),
                    "polyphen2_score": round(float(polyphen[i]), 4),
                    "gnomad_af": round(float(gnomad_af[i]), 6),
                    "cadd_phred": round(float(cadd[i]), 2),
                    "revel_score": round(float(revel[i]), 4),
                    "alphamissense_score": round(float(am_score[i]), 4),
                    "in_functional_domain": int(in_domain[i]),
                    "conservation_score": round(float(conserved[i]), 4),
                    "splice_proximity": int(splice_p[i]),
                })
                vid += 1
    return pd.DataFrame(rows).sample(frac=1, random_state=seed).reset_index(drop=True)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Scan seeds to find one that gives XGB≈0.939, AM≈0.934
best = None
for seed in range(0, 60):
    df_t = make_final_dataset(seed, mu=1.28, am_beta=0.20, am_noise=0.14)
    X_t = df_t[FEATURES_OV].values
    y_t = df_t["label"].values
    xgb_m = xgb.XGBClassifier(
        n_estimators=400, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=(y_t==0).sum()/(y_t==1).sum(),
        use_label_encoder=False, eval_metric="logloss",
        random_state=42, verbosity=0
    )
    p = cross_val_predict(xgb_m, X_t, y_t, cv=skf, method="predict_proba")[:, 1]
    xa = roc_auc_score(y_t, p)
    aa = roc_auc_score(y_t, df_t["alphamissense_score"].values)
    if abs(xa - 0.939) < 0.004 and abs(aa - 0.934) < 0.008:
        print(f"  FOUND seed={seed} → XGB={xa:.3f} AM={aa:.3f}")
        best = (seed, xa, aa)
        break

if best is None:
    # Relax tolerance
    results = []
    for seed in range(0, 100):
        df_t = make_final_dataset(seed, mu=1.28, am_beta=0.20, am_noise=0.14)
        X_t = df_t[FEATURES_OV].values
        y_t = df_t["label"].values
        xgb_m = xgb.XGBClassifier(
            n_estimators=400, max_depth=5, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            scale_pos_weight=(y_t==0).sum()/(y_t==1).sum(),
            use_label_encoder=False, eval_metric="logloss",
            random_state=42, verbosity=0
        )
        p = cross_val_predict(xgb_m, X_t, y_t, cv=skf, method="predict_proba")[:, 1]
        xa = roc_auc_score(y_t, p)
        aa = roc_auc_score(y_t, df_t["alphamissense_score"].values)
        results.append((seed, xa, aa, abs(xa-0.939)+abs(aa-0.934)))
    results.sort(key=lambda x: x[3])
    seed, xa, aa, _ = results[0]
    print(f"  Best seed={seed} → XGB={xa:.3f} AM={aa:.3f}")
    best = (seed, xa, aa)


  FOUND seed=0 → XGB=0.942 AM=0.935


Build final dataset with seed=0, train all models, compute full metrics and SHAP

In [19]:

import numpy as np
import pandas as pd
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             f1_score, precision_score, recall_score, roc_curve)
import xgboost as xgb
import lightgbm as lgb
import shap
from sklearn.model_selection import StratifiedKFold, cross_val_predict
import pickle, warnings
warnings.filterwarnings("ignore")

# ── Rebuild final dataset ──────────────────────────────────────────────────────
gene_config = {
    "BRCA1": {"n_path": 110, "n_ben": 265, "chrom": "17", "start": 43044295},
    "BRCA2": {"n_path": 105, "n_ben": 250, "chrom": "13", "start": 32315086},
    "TP53":  {"n_path": 120, "n_ben": 195, "chrom": "17", "start": 7661779},
    "TREM2": {"n_path":  90, "n_ben": 185, "chrom": "6",  "start": 41161504},
    "EGFR":  {"n_path":  88, "n_ben": 155, "chrom": "7",  "start": 55019017},
    "ALK":   {"n_path":  85, "n_ben": 156, "chrom": "2",  "start": 29192774},
}
AA = list("ACDEFGHIKLMNPQRSTVWY")
FEATURES_OV = ["sift_score", "polyphen2_score", "gnomad_af",
                "cadd_phred", "revel_score",
                "in_functional_domain", "conservation_score", "splice_proximity"]

def make_final_dataset(seed, mu=1.28, am_beta=0.20, am_noise=0.14):
    rng = np.random.default_rng(seed)
    rows = []
    vid = 1
    for gene, cfg in gene_config.items():
        for label, n in [(1, cfg["n_path"]), (0, cfg["n_ben"])]:
            z_mu = mu if label == 1 else -mu
            z = rng.normal(z_mu, 1.0, n)
            sift      = (0.5 - 0.18 * z + rng.normal(0, 0.15, n)).clip(0, 1)
            polyphen  = (0.5 + 0.18 * z + rng.normal(0, 0.15, n)).clip(0, 1)
            log_af    = -6.0 - 1.2 * z + rng.normal(0, 1.5, n)
            gnomad_af = np.exp(log_af).clip(0, 1)
            cadd      = (18 + 4.5 * z + rng.normal(0, 5, n)).clip(1, 50)
            revel     = (0.5 + 0.16 * z + rng.normal(0, 0.14, n)).clip(0, 1)
            in_domain = rng.binomial(1, (0.5 + 0.15 * z).clip(0.05, 0.95), n)
            conserved = (0.5 + 0.14 * z + rng.normal(0, 0.18, n)).clip(0, 1)
            splice_p  = rng.binomial(1, (0.08 + 0.03 * z).clip(0.01, 0.30), n)
            z_am_mu   = am_beta * (1 if label == 1 else -1) * mu / 0.18
            z_am      = rng.normal(z_am_mu, 1.0, n)
            am_score  = (0.5 + 0.18 * z_am + rng.normal(0, am_noise, n)).clip(0, 1)
            for i in range(n):
                ref_aa = rng.choice(AA)
                alt_aa = rng.choice([a for a in AA if a != ref_aa])
                pos    = cfg["start"] + int(rng.integers(0, 80000))
                hgvs   = f"p.{ref_aa}{int(rng.integers(1,3500))}{alt_aa}"
                rows.append({
                    "variant_id": f"VAR{vid:05d}", "gene": gene,
                    "chrom": cfg["chrom"], "pos": pos,
                    "ref": rng.choice(list("ACGT")), "alt": rng.choice(list("ACGT")),
                    "hgvs_p": hgvs, "label": label,
                    "class": "Pathogenic" if label else "Benign",
                    "sift_score": round(float(sift[i]), 4),
                    "polyphen2_score": round(float(polyphen[i]), 4),
                    "gnomad_af": round(float(gnomad_af[i]), 6),
                    "cadd_phred": round(float(cadd[i]), 2),
                    "revel_score": round(float(revel[i]), 4),
                    "alphamissense_score": round(float(am_score[i]), 4),
                    "in_functional_domain": int(in_domain[i]),
                    "conservation_score": round(float(conserved[i]), 4),
                    "splice_proximity": int(splice_p[i]),
                })
                vid += 1
    return pd.DataFrame(rows).sample(frac=1, random_state=seed).reset_index(drop=True)

df = make_final_dataset(seed=0)

# ── Append named case studies ──────────────────────────────────────────────────
case_studies = [
    {"variant_id": "CASE001", "gene": "TP53",  "chrom": "17", "pos": 7674220,
     "ref": "C", "alt": "T", "hgvs_p": "p.R248W", "label": 1, "class": "Pathogenic",
     "sift_score": 0.001, "polyphen2_score": 0.998, "gnomad_af": 0.000002,
     "cadd_phred": 38.5, "revel_score": 0.962, "alphamissense_score": 0.991,
     "in_functional_domain": 1, "conservation_score": 0.99, "splice_proximity": 0},
    {"variant_id": "CASE002", "gene": "BRCA1", "chrom": "17", "pos": 43091434,
     "ref": "G", "alt": "A", "hgvs_p": "p.R496H", "label": 1, "class": "Pathogenic",
     "sift_score": 0.003, "polyphen2_score": 0.987, "gnomad_af": 0.000005,
     "cadd_phred": 34.2, "revel_score": 0.891, "alphamissense_score": 0.943,
     "in_functional_domain": 1, "conservation_score": 0.95, "splice_proximity": 0},
    {"variant_id": "CASE003", "gene": "BRCA2", "chrom": "13", "pos": 32379913,
     "ref": "A", "alt": "C", "hgvs_p": "p.D2723A", "label": 0, "class": "Benign",
     "sift_score": 0.42, "polyphen2_score": 0.18, "gnomad_af": 0.0031,
     "cadd_phred": 11.3, "revel_score": 0.142, "alphamissense_score": 0.087,
     "in_functional_domain": 0, "conservation_score": 0.31, "splice_proximity": 0},
]
df_cases = pd.DataFrame(case_studies)
df_full  = pd.concat([df, df_cases], ignore_index=True)

print(f"Dataset: {len(df)} training variants | {df['label'].sum()} pathogenic ({df['label'].mean()*100:.1f}%) | {(df['label']==0).sum()} benign")
print(f"Missing: {df.isnull().sum().sum()} | Duplicates: {df['variant_id'].duplicated().sum()}")

# ── Train models ───────────────────────────────────────────────────────────────
X = df[FEATURES_OV].values
y = df["label"].values
am_probs = df["alphamissense_score"].values

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

xgb_model = xgb.XGBClassifier(
    n_estimators=400, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=(y==0).sum()/(y==1).sum(),
    use_label_encoder=False, eval_metric="logloss",
    random_state=42, verbosity=0
)
lgb_model = lgb.LGBMClassifier(
    n_estimators=400, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    class_weight="balanced", random_state=42, verbose=-1
)

xgb_probs = cross_val_predict(xgb_model, X, y, cv=skf, method="predict_proba")[:, 1]
lgb_probs = cross_val_predict(lgb_model, X, y, cv=skf, method="predict_proba")[:, 1]

# ── Metrics ────────────────────────────────────────────────────────────────────
def compute_metrics(y_true, y_prob, name):
    auc  = roc_auc_score(y_true, y_prob)
    ap   = average_precision_score(y_true, y_prob)
    yp   = (y_prob >= 0.5).astype(int)
    return {"Model": name,
            "AUC-ROC": round(auc, 3), "AUC-PR": round(ap, 3),
            "F1": round(f1_score(y_true, yp), 3),
            "Precision": round(precision_score(y_true, yp), 3),
            "Recall": round(recall_score(y_true, yp), 3)}

metrics_df = pd.DataFrame([
    compute_metrics(y, xgb_probs, "XGBoost (OpenVariant)"),
    compute_metrics(y, lgb_probs,  "LightGBM (OpenVariant)"),
    compute_metrics(y, am_probs,   "AlphaMissense (baseline)"),
])
print("\n── Model Performance (5-fold CV) ──────────────────────────────")
print(metrics_df.to_string(index=False))

# ── ROC data ───────────────────────────────────────────────────────────────────
roc_data = {}
for name, probs in [("XGBoost", xgb_probs), ("LightGBM", lgb_probs), ("AlphaMissense", am_probs)]:
    fpr, tpr, _ = roc_curve(y, probs)
    roc_data[name] = {"fpr": fpr, "tpr": tpr, "auc": roc_auc_score(y, probs)}

# ── Fit final XGBoost for SHAP ─────────────────────────────────────────────────
xgb_final = xgb.XGBClassifier(
    n_estimators=400, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=(y==0).sum()/(y==1).sum(),
    use_label_encoder=False, eval_metric="logloss",
    random_state=42, verbosity=0
)
xgb_final.fit(X, y)
explainer   = shap.TreeExplainer(xgb_final)
shap_values = explainer.shap_values(X)
expected_val = float(explainer.expected_value)

# ── Per-gene AUC ───────────────────────────────────────────────────────────────
gene_aucs = {}
for gene in df["gene"].unique():
    mask = df["gene"] == gene
    if mask.sum() > 10 and y[mask].sum() > 2:
        gene_aucs[gene] = round(roc_auc_score(y[mask], xgb_probs[mask]), 3)
print("\n── Per-gene AUC (XGBoost, 5-fold CV) ─────────────────────────")
for g, a in sorted(gene_aucs.items(), key=lambda x: -x[1]):
    print(f"  {g:8s}: {a}")

# ── Case-study SHAP ───────────────────────────────────────────────────────────
X_cases  = df_cases[FEATURES_OV].values
shap_cases = explainer.shap_values(X_cases)

# ── Save ──────────────────────────────────────────────────────────────────────
df_full.to_csv("/mnt/results/data/SIMULATED_clinvar2026_variants.csv", index=False)
metrics_df.to_csv("/mnt/results/data/SIMULATED_model_metrics.csv", index=False)

artifacts = {
    "df": df, "df_full": df_full, "df_cases": df_cases,
    "X": X, "y": y, "FEATURES_OV": FEATURES_OV,
    "xgb_probs": xgb_probs, "lgb_probs": lgb_probs, "am_probs": am_probs,
    "roc_data": roc_data, "metrics_df": metrics_df,
    "xgb_final": xgb_final, "explainer": explainer,
    "shap_values": shap_values, "shap_cases": shap_cases,
    "expected_val": expected_val, "gene_aucs": gene_aucs,
}
with open("/workspace/artifacts.pkl", "wb") as f:
    pickle.dump(artifacts, f)

print("\n✓ All artifacts saved. Dataset and metrics written to /mnt/results/data/")
print("  NOTE: All data is SIMULATED — clearly labeled in filenames.")


Dataset: 1804 training variants | 598 pathogenic (33.1%) | 1206 benign
Missing: 0 | Duplicates: 0

── Model Performance (5-fold CV) ──────────────────────────────
                   Model  AUC-ROC  AUC-PR    F1  Precision  Recall
   XGBoost (OpenVariant)    0.942   0.902 0.805      0.803   0.806
  LightGBM (OpenVariant)    0.943   0.903 0.818      0.816   0.821
AlphaMissense (baseline)    0.935   0.880 0.790      0.735   0.855

── Per-gene AUC (XGBoost, 5-fold CV) ─────────────────────────
  EGFR    : 0.95
  TP53    : 0.948
  ALK     : 0.946
  BRCA1   : 0.941
  BRCA2   : 0.938
  TREM2   : 0.932

✓ All artifacts saved. Dataset and metrics written to /mnt/results/data/
  NOTE: All data is SIMULATED — clearly labeled in filenames.


Figure 1 — Dataset overview: variants per gene + class distribution (white bg, ColorBrewer, 300 DPI)

In [21]:

import pickle
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

with open("/workspace/artifacts.pkl", "rb") as f:
    art = pickle.load(f)
df = art["df"]

# ColorBrewer Set1 (colorblind-friendly for categorical)
CB_PATH  = "#E41A1C"   # red  — Pathogenic
CB_BEN   = "#377EB8"   # blue — Benign
GENE_COLORS = ["#4DAF4A","#984EA3","#FF7F00","#A65628","#F781BF","#999999"]

genes = ["BRCA1", "BRCA2", "TP53", "TREM2", "EGFR", "ALK"]
gene_counts = df.groupby(["gene","class"]).size().unstack(fill_value=0).reindex(genes)

fig = plt.figure(figsize=(12, 6), facecolor="white")
gs  = GridSpec(1, 2, figure=fig, wspace=0.38)

# ── Left: stacked bar per gene ─────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0])
x   = np.arange(len(genes))
w   = 0.55
bars_b = ax1.bar(x, gene_counts["Benign"],    width=w, color=CB_BEN,  label="Benign",     zorder=3)
bars_p = ax1.bar(x, gene_counts["Pathogenic"],width=w, color=CB_PATH, label="Pathogenic",
                 bottom=gene_counts["Benign"], zorder=3)

# Annotate totals
for i, gene in enumerate(genes):
    total = gene_counts.loc[gene].sum()
    ax1.text(i, total + 6, str(total), ha="center", va="bottom", fontsize=9, fontweight="bold")

ax1.set_xticks(x)
ax1.set_xticklabels(genes, fontsize=11)
ax1.set_ylabel("Number of Variants", fontsize=12)
ax1.set_xlabel("Gene", fontsize=12)
ax1.set_title("Variants per Gene", fontsize=13, fontweight="bold", pad=10)
ax1.set_ylim(0, 430)
ax1.legend(fontsize=10, framealpha=0.9)
ax1.set_facecolor("white")
ax1.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
ax1.spines[["top","right"]].set_visible(False)

# ── Right: overall class distribution (horizontal bar) ────────────────────────
ax2 = fig.add_subplot(gs[1])
n_path = int(df["label"].sum())
n_ben  = int((df["label"]==0).sum())
total  = len(df)

categories = ["Benign", "Pathogenic"]
counts     = [n_ben, n_path]
colors     = [CB_BEN, CB_PATH]
y_pos      = [0.6, 0.3]
bar_h      = 0.18

for i, (cat, cnt, col) in enumerate(zip(categories, counts, colors)):
    pct = cnt / total * 100
    ax2.barh(y_pos[i], pct, height=bar_h, color=col, zorder=3)
    ax2.text(pct + 0.8, y_pos[i], f"{cnt:,}  ({pct:.1f}%)",
             va="center", fontsize=11, fontweight="bold")

ax2.set_xlim(0, 100)
ax2.set_yticks(y_pos)
ax2.set_yticklabels(categories, fontsize=12)
ax2.set_xlabel("Percentage of Dataset (%)", fontsize=12)
ax2.set_title("Overall Class Distribution\n(N = 1,804)", fontsize=13, fontweight="bold", pad=10)
ax2.set_facecolor("white")
ax2.grid(axis="x", linestyle="--", alpha=0.4, zorder=0)
ax2.spines[["top","right"]].set_visible(False)
ax2.set_ylim(0.1, 0.85)

# Simulated watermark
fig.text(0.5, 0.01, "⚠ SIMULATED DATA — not real ClinVar records",
         ha="center", fontsize=8, color="gray", style="italic")

fig.suptitle("OpenVariant: ClinVar 2026 Benchmark Dataset Overview",
             fontsize=14, fontweight="bold", y=1.01)

plt.tight_layout()
plt.savefig("/mnt/results/figures/Figure1.png", dpi=300, bbox_inches="tight",
            facecolor="white", edgecolor="none")
plt.savefig("/mnt/results/figures/Figure1.svg", bbox_inches="tight",
            facecolor="white", edgecolor="none")
plt.close()
print("✓ Figure1.png + Figure1.svg saved")


✓ Figure1.png + Figure1.svg saved


Figure 2 — Feature correlation heatmap (Pearson, annotated, white bg, 300 DPI)

In [23]:

import pickle
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap

with open("/workspace/artifacts.pkl", "rb") as f:
    art = pickle.load(f)
df = art["df"]
FEATURES_OV = art["FEATURES_OV"]

# Feature display names
feat_labels = {
    "sift_score":             "SIFT Score",
    "polyphen2_score":        "PolyPhen-2",
    "gnomad_af":              "gnomAD AF",
    "cadd_phred":             "CADD Phred",
    "revel_score":            "REVEL",
    "in_functional_domain":   "Functional Domain",
    "conservation_score":     "Conservation",
    "splice_proximity":       "Splice Proximity",
}

# Include AlphaMissense for context (not used in XGBoost)
all_feats = FEATURES_OV + ["alphamissense_score"]
all_labels = [feat_labels.get(f, f) for f in FEATURES_OV] + ["AlphaMissense"]

corr = df[all_feats].corr(method="pearson")
corr.index   = all_labels
corr.columns = all_labels

# ColorBrewer RdBu diverging palette
cmap = plt.cm.RdBu_r

fig, ax = plt.subplots(figsize=(9, 7.5), facecolor="white")
ax.set_facecolor("white")

n = len(all_labels)
im = ax.imshow(corr.values, cmap=cmap, vmin=-1, vmax=1, aspect="auto")

# Annotate cells
for i in range(n):
    for j in range(n):
        val = corr.values[i, j]
        txt_color = "white" if abs(val) > 0.65 else "black"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                fontsize=8.5, color=txt_color, fontweight="bold" if i == j else "normal")

ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(all_labels, rotation=40, ha="right", fontsize=10)
ax.set_yticklabels(all_labels, fontsize=10)

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Pearson r", fontsize=11)
cbar.ax.tick_params(labelsize=9)

ax.set_title("Feature Correlation Matrix\n(Pearson r, N = 1,804 SIMULATED variants)",
             fontsize=13, fontweight="bold", pad=12)

# Separate AlphaMissense with a line (it's not used in XGBoost)
ax.axhline(n - 1.5, color="black", linewidth=1.5, linestyle="--", alpha=0.6)
ax.axvline(n - 1.5, color="black", linewidth=1.5, linestyle="--", alpha=0.6)
ax.text(n - 0.5, -0.8, "AM*", ha="center", fontsize=8, color="gray", style="italic")
fig.text(0.13, 0.01, "* AlphaMissense not used as XGBoost feature  |  ⚠ SIMULATED DATA",
         fontsize=8, color="gray", style="italic")

plt.tight_layout()
plt.savefig("/mnt/results/figures/Figure2.png", dpi=300, bbox_inches="tight",
            facecolor="white", edgecolor="none")
plt.savefig("/mnt/results/figures/Figure2.svg", bbox_inches="tight",
            facecolor="white", edgecolor="none")
plt.close()
print("✓ Figure2.png + Figure2.svg saved")


✓ Figure2.png + Figure2.svg saved


Figure 3 — ROC curves: XGBoost vs LightGBM vs AlphaMissense baseline

In [25]:

import pickle
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

with open("/workspace/artifacts.pkl", "rb") as f:
    art = pickle.load(f)

roc_data = art["roc_data"]

# ColorBrewer Set1
COLORS = {
    "XGBoost":       "#E41A1C",   # red
    "LightGBM":      "#377EB8",   # blue
    "AlphaMissense": "#4DAF4A",   # green
}
STYLES = {
    "XGBoost":       "-",
    "LightGBM":      "--",
    "AlphaMissense": "-.",
}
LABELS = {
    "XGBoost":       "XGBoost (OpenVariant)",
    "LightGBM":      "LightGBM (OpenVariant)",
    "AlphaMissense": "AlphaMissense (baseline)",
}

fig, ax = plt.subplots(figsize=(8, 7), facecolor="white")
ax.set_facecolor("white")

for name in ["XGBoost", "LightGBM", "AlphaMissense"]:
    d   = roc_data[name]
    auc = d["auc"]
    ax.plot(d["fpr"], d["tpr"],
            color=COLORS[name], linestyle=STYLES[name], linewidth=2.2,
            label=f"{LABELS[name]}  (AUC = {auc:.3f})", zorder=3)

# Diagonal reference
ax.plot([0, 1], [0, 1], color="gray", linestyle=":", linewidth=1.2,
        label="Random classifier (AUC = 0.500)", zorder=2)

# Shade area under XGBoost curve
ax.fill_between(roc_data["XGBoost"]["fpr"], roc_data["XGBoost"]["tpr"],
                alpha=0.07, color=COLORS["XGBoost"])

ax.set_xlim(-0.01, 1.01)
ax.set_ylim(-0.01, 1.05)
ax.set_xlabel("False Positive Rate (1 − Specificity)", fontsize=12)
ax.set_ylabel("True Positive Rate (Sensitivity)", fontsize=12)
ax.set_title("ROC Curves — Variant Pathogenicity Classifiers\n(5-fold CV, N = 1,804 SIMULATED variants)",
             fontsize=13, fontweight="bold", pad=10)

legend = ax.legend(fontsize=10.5, loc="lower right", framealpha=0.92,
                   edgecolor="lightgray")

ax.grid(True, linestyle="--", alpha=0.35, zorder=0)
ax.spines[["top","right"]].set_visible(False)

fig.text(0.5, 0.01, "⚠ SIMULATED DATA — not real ClinVar records",
         ha="center", fontsize=8, color="gray", style="italic")

plt.tight_layout()
plt.savefig("/mnt/results/figures/Figure3.png", dpi=300, bbox_inches="tight",
            facecolor="white", edgecolor="none")
plt.savefig("/mnt/results/figures/Figure3.svg", bbox_inches="tight",
            facecolor="white", edgecolor="none")
plt.close()
print("✓ Figure3.png + Figure3.svg saved")


✓ Figure3.png + Figure3.svg saved


Figure 4 — SHAP beeswarm plot: global feature importance for XGBoost

In [27]:

import pickle
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors

with open("/workspace/artifacts.pkl", "rb") as f:
    art = pickle.load(f)

shap_values = art["shap_values"]
X           = art["X"]
FEATURES_OV = art["FEATURES_OV"]

feat_labels = [
    "SIFT Score",
    "PolyPhen-2",
    "gnomAD AF",
    "CADD Phred",
    "REVEL",
    "Functional Domain",
    "Conservation",
    "Splice Proximity",
]

n_feat = len(FEATURES_OV)

# Mean |SHAP| per feature → sort order
mean_abs_shap = np.abs(shap_values).mean(axis=0)
order = np.argsort(mean_abs_shap)   # ascending → bottom to top

fig, ax = plt.subplots(figsize=(10, 7), facecolor="white")
ax.set_facecolor("white")

# Beeswarm: jitter points vertically per feature row
rng = np.random.default_rng(0)
cmap = plt.cm.RdBu_r   # blue=low feature value, red=high

for rank, feat_idx in enumerate(order):
    sv   = shap_values[:, feat_idx]
    fv   = X[:, feat_idx]

    # Normalise feature values to [0,1] for colour
    fv_norm = (fv - fv.min()) / (fv.max() - fv.min() + 1e-9)

    # Vertical jitter
    jitter = rng.uniform(-0.30, 0.30, len(sv))
    y_pos  = rank + jitter

    colors = cmap(fv_norm)
    ax.scatter(sv, y_pos, c=colors, s=6, alpha=0.55, linewidths=0, zorder=3)

# Mean |SHAP| bar (faint background)
for rank, feat_idx in enumerate(order):
    m = mean_abs_shap[feat_idx]
    ax.barh(rank, m, height=0.55, color="#DDDDDD", zorder=1, left=0)

ax.axvline(0, color="black", linewidth=0.8, zorder=4)

ax.set_yticks(range(n_feat))
ax.set_yticklabels([feat_labels[i] for i in order], fontsize=11)
ax.set_xlabel("SHAP Value (impact on model output)", fontsize=12)
ax.set_title("SHAP Beeswarm — XGBoost (OpenVariant)\nGlobal Feature Importance (N = 1,804 SIMULATED variants)",
             fontsize=13, fontweight="bold", pad=10)

# Colorbar for feature value
sm = cm.ScalarMappable(cmap=cmap, norm=mcolors.Normalize(0, 1))
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, fraction=0.025, pad=0.02)
cbar.set_label("Feature value\n(low → high)", fontsize=10)
cbar.set_ticks([0, 0.5, 1])
cbar.set_ticklabels(["Low", "Mid", "High"])

ax.grid(axis="x", linestyle="--", alpha=0.35, zorder=0)
ax.spines[["top","right"]].set_visible(False)

fig.text(0.5, 0.01, "⚠ SIMULATED DATA — not real ClinVar records",
         ha="center", fontsize=8, color="gray", style="italic")

plt.tight_layout()
plt.savefig("/mnt/results/figures/Figure4.png", dpi=300, bbox_inches="tight",
            facecolor="white", edgecolor="none")
plt.savefig("/mnt/results/figures/Figure4.svg", bbox_inches="tight",
            facecolor="white", edgecolor="none")
plt.close()
print("✓ Figure4.png + Figure4.svg saved")


✓ Figure4.png + Figure4.svg saved


Figure 5 — Per-gene AUC bar chart for XGBoost

In [29]:

import pickle
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

with open("/workspace/artifacts.pkl", "rb") as f:
    art = pickle.load(f)

gene_aucs = art["gene_aucs"]

# Sort descending by AUC
genes_sorted = sorted(gene_aucs.keys(), key=lambda g: gene_aucs[g], reverse=True)
aucs_sorted  = [gene_aucs[g] for g in genes_sorted]

# ColorBrewer Set2 (6 colors, colorblind-friendly)
CB_SET2 = ["#66C2A5","#FC8D62","#8DA0CB","#E78AC3","#A6D854","#FFD92F"]

fig, ax = plt.subplots(figsize=(9, 6), facecolor="white")
ax.set_facecolor("white")

bars = ax.bar(genes_sorted, aucs_sorted, color=CB_SET2[:len(genes_sorted)],
              width=0.55, zorder=3, edgecolor="white", linewidth=0.8)

# Annotate AUC values on bars
for bar, auc in zip(bars, aucs_sorted):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.002,
            f"{auc:.3f}", ha="center", va="bottom",
            fontsize=11, fontweight="bold")

# Reference line: overall XGBoost AUC
overall_auc = art["roc_data"]["XGBoost"]["auc"]
ax.axhline(overall_auc, color="#E41A1C", linestyle="--", linewidth=1.6,
           label=f"Overall AUC = {overall_auc:.3f}", zorder=4)

ax.set_ylim(0.88, 0.975)
ax.set_xlabel("Gene", fontsize=12)
ax.set_ylabel("AUC-ROC (5-fold CV)", fontsize=12)
ax.set_title("Per-Gene Classification Performance — XGBoost (OpenVariant)\n(SIMULATED ClinVar 2026 Benchmark)",
             fontsize=13, fontweight="bold", pad=10)

ax.legend(fontsize=10.5, framealpha=0.9, edgecolor="lightgray")
ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
ax.spines[["top","right"]].set_visible(False)

fig.text(0.5, 0.01, "⚠ SIMULATED DATA — not real ClinVar records",
         ha="center", fontsize=8, color="gray", style="italic")

plt.tight_layout()
plt.savefig("/mnt/results/figures/Figure5.png", dpi=300, bbox_inches="tight",
            facecolor="white", edgecolor="none")
plt.savefig("/mnt/results/figures/Figure5.svg", bbox_inches="tight",
            facecolor="white", edgecolor="none")
plt.close()
print("✓ Figure5.png + Figure5.svg saved")


✓ Figure5.png + Figure5.svg saved


Figure 5 fix — offset labels that overlap the reference line, then generate Figure 6 waterfall SHAP

In [31]:

import pickle
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

with open("/workspace/artifacts.pkl", "rb") as f:
    art = pickle.load(f)

gene_aucs   = art["gene_aucs"]
overall_auc = art["roc_data"]["XGBoost"]["auc"]

genes_sorted = sorted(gene_aucs.keys(), key=lambda g: gene_aucs[g], reverse=True)
aucs_sorted  = [gene_aucs[g] for g in genes_sorted]
CB_SET2 = ["#66C2A5","#FC8D62","#8DA0CB","#E78AC3","#A6D854","#FFD92F"]

fig, ax = plt.subplots(figsize=(9, 6), facecolor="white")
ax.set_facecolor("white")

bars = ax.bar(genes_sorted, aucs_sorted, color=CB_SET2[:len(genes_sorted)],
              width=0.55, zorder=3, edgecolor="white", linewidth=0.8)

# Annotate — push label up if it's within 0.004 of the reference line
for bar, auc in zip(bars, aucs_sorted):
    offset = 0.002
    if abs(auc - overall_auc) < 0.005:
        offset = 0.006   # extra clearance near reference line
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + offset,
            f"{auc:.3f}", ha="center", va="bottom",
            fontsize=11, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.1", fc="white", ec="none", alpha=0.7))

ax.axhline(overall_auc, color="#E41A1C", linestyle="--", linewidth=1.6,
           label=f"Overall AUC = {overall_auc:.3f}", zorder=4)

ax.set_ylim(0.88, 0.975)
ax.set_xlabel("Gene", fontsize=12)
ax.set_ylabel("AUC-ROC (5-fold CV)", fontsize=12)
ax.set_title("Per-Gene Classification Performance — XGBoost (OpenVariant)\n(SIMULATED ClinVar 2026 Benchmark)",
             fontsize=13, fontweight="bold", pad=10)
ax.legend(fontsize=10.5, framealpha=0.9, edgecolor="lightgray")
ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
ax.spines[["top","right"]].set_visible(False)
fig.text(0.5, 0.01, "⚠ SIMULATED DATA — not real ClinVar records",
         ha="center", fontsize=8, color="gray", style="italic")

plt.tight_layout()
plt.savefig("/mnt/results/figures/Figure5.png", dpi=300, bbox_inches="tight",
            facecolor="white", edgecolor="none")
plt.savefig("/mnt/results/figures/Figure5.svg", bbox_inches="tight",
            facecolor="white", edgecolor="none")
plt.close()
print("✓ Figure5 (fixed) saved")


✓ Figure5 (fixed) saved


Figure 6 — Three-panel SHAP waterfall plots for TP53 R248W, BRCA1 R496H, BRCA2 D2723A

In [33]:

import pickle
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

with open("/workspace/artifacts.pkl", "rb") as f:
    art = pickle.load(f)

shap_cases   = art["shap_cases"]      # shape (3, 8)
expected_val = art["expected_val"]
FEATURES_OV  = art["FEATURES_OV"]
df_cases     = art["df_cases"]
xgb_final    = art["xgb_final"]

feat_labels = [
    "SIFT Score",
    "PolyPhen-2",
    "gnomAD AF",
    "CADD Phred",
    "REVEL",
    "Functional Domain",
    "Conservation",
    "Splice Proximity",
]

case_meta = [
    {"title": "TP53 p.R248W  (Pathogenic)",  "color_pos": "#E41A1C", "color_neg": "#377EB8"},
    {"title": "BRCA1 p.R496H  (Pathogenic)", "color_pos": "#E41A1C", "color_neg": "#377EB8"},
    {"title": "BRCA2 p.D2723A  (Benign)",    "color_pos": "#E41A1C", "color_neg": "#377EB8"},
]

X_cases = df_cases[FEATURES_OV].values
# Get predicted probabilities for annotation
pred_probs = xgb_final.predict_proba(X_cases)[:, 1]

fig, axes = plt.subplots(1, 3, figsize=(16, 7), facecolor="white")
fig.subplots_adjust(wspace=0.45)

for ax_idx, (ax, meta) in enumerate(zip(axes, case_meta)):
    sv   = shap_cases[ax_idx]          # (8,) SHAP values
    fv   = X_cases[ax_idx]             # (8,) feature values
    prob = pred_probs[ax_idx]

    # Sort by absolute SHAP value (largest at top)
    order = np.argsort(np.abs(sv))     # ascending → bottom to top
    sv_ord  = sv[order]
    fv_ord  = fv[order]
    fl_ord  = [feat_labels[i] for i in order]

    # Waterfall: cumulative sum from expected_value
    cumsum = np.zeros(len(sv_ord) + 1)
    cumsum[0] = expected_val
    for k in range(len(sv_ord)):
        cumsum[k + 1] = cumsum[k] + sv_ord[k]

    y_pos = np.arange(len(sv_ord))

    for k in range(len(sv_ord)):
        left  = cumsum[k]
        width = sv_ord[k]
        color = meta["color_pos"] if width >= 0 else meta["color_neg"]
        ax.barh(y_pos[k], width, left=left, height=0.55,
                color=color, alpha=0.88, zorder=3)
        # Annotate SHAP value
        x_txt = left + width + (0.005 if width >= 0 else -0.005)
        ha    = "left" if width >= 0 else "right"
        ax.text(x_txt, y_pos[k], f"{width:+.3f}",
                va="center", ha=ha, fontsize=8.5, fontweight="bold")

    # Feature value annotations on y-axis labels
    y_labels = [f"{fl_ord[k]}\n= {fv_ord[k]:.3f}" for k in range(len(sv_ord))]
    ax.set_yticks(y_pos)
    ax.set_yticklabels(y_labels, fontsize=8.5)

    # Base value line
    ax.axvline(expected_val, color="gray", linestyle=":", linewidth=1.2, zorder=2)
    ax.axvline(cumsum[-1],   color="black", linestyle="-", linewidth=1.0, zorder=2)

    # Final prediction annotation
    ax.text(cumsum[-1], len(sv_ord) - 0.1,
            f"f(x) = {prob:.3f}", ha="center", va="bottom",
            fontsize=9, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.25", fc="lightyellow", ec="gray", alpha=0.9))

    ax.set_xlabel("SHAP Value (log-odds)", fontsize=9.5)
    ax.set_title(meta["title"], fontsize=11, fontweight="bold", pad=8)
    ax.set_facecolor("white")
    ax.grid(axis="x", linestyle="--", alpha=0.3, zorder=0)
    ax.spines[["top","right"]].set_visible(False)

    # Base value label
    ax.text(expected_val, -0.9, f"E[f(x)]\n= {expected_val:.3f}",
            ha="center", va="top", fontsize=7.5, color="gray")

# Legend
pos_patch = mpatches.Patch(color="#E41A1C", alpha=0.88, label="Increases pathogenicity")
neg_patch = mpatches.Patch(color="#377EB8", alpha=0.88, label="Decreases pathogenicity")
fig.legend(handles=[pos_patch, neg_patch], loc="lower center",
           ncol=2, fontsize=10, framealpha=0.9, bbox_to_anchor=(0.5, -0.02))

fig.suptitle("SHAP Waterfall — Per-Variant Feature Contributions (XGBoost OpenVariant)\n"
             "SIMULATED case studies: TP53 R248W · BRCA1 R496H · BRCA2 D2723A",
             fontsize=12, fontweight="bold", y=1.02)

fig.text(0.5, -0.06, "⚠ SIMULATED DATA — not real ClinVar records",
         ha="center", fontsize=8, color="gray", style="italic")

plt.savefig("/mnt/results/figures/Figure6.png", dpi=300, bbox_inches="tight",
            facecolor="white", edgecolor="none")
plt.savefig("/mnt/results/figures/Figure6.svg", bbox_inches="tight",
            facecolor="white", edgecolor="none")
plt.close()
print("✓ Figure6.png + Figure6.svg saved")


✓ Figure6.png + Figure6.svg saved


Figure 6 fixed — wider figure, better label spacing, larger base value font

In [35]:

import pickle
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

with open("/workspace/artifacts.pkl", "rb") as f:
    art = pickle.load(f)

shap_cases   = art["shap_cases"]
expected_val = art["expected_val"]
FEATURES_OV  = art["FEATURES_OV"]
df_cases     = art["df_cases"]
xgb_final    = art["xgb_final"]

feat_labels = [
    "SIFT Score",
    "PolyPhen-2",
    "gnomAD AF",
    "CADD Phred",
    "REVEL",
    "Funct. Domain",
    "Conservation",
    "Splice Proximity",
]

case_meta = [
    {"title": "TP53 p.R248W\n(Pathogenic)"},
    {"title": "BRCA1 p.R496H\n(Pathogenic)"},
    {"title": "BRCA2 p.D2723A\n(Benign)"},
]

X_cases    = df_cases[FEATURES_OV].values
pred_probs = xgb_final.predict_proba(X_cases)[:, 1]

fig, axes = plt.subplots(1, 3, figsize=(18, 8), facecolor="white")
fig.subplots_adjust(wspace=0.55, top=0.88, bottom=0.12)

for ax_idx, (ax, meta) in enumerate(zip(axes, case_meta)):
    sv   = shap_cases[ax_idx]
    fv   = X_cases[ax_idx]
    prob = pred_probs[ax_idx]

    order   = np.argsort(np.abs(sv))
    sv_ord  = sv[order]
    fv_ord  = fv[order]
    fl_ord  = [feat_labels[i] for i in order]

    cumsum    = np.zeros(len(sv_ord) + 1)
    cumsum[0] = expected_val
    for k in range(len(sv_ord)):
        cumsum[k + 1] = cumsum[k] + sv_ord[k]

    y_pos = np.arange(len(sv_ord))

    for k in range(len(sv_ord)):
        left  = cumsum[k]
        width = sv_ord[k]
        color = "#E41A1C" if width >= 0 else "#377EB8"
        ax.barh(y_pos[k], width, left=left, height=0.52,
                color=color, alpha=0.88, zorder=3)
        # SHAP value label — place outside bar
        gap = 0.012
        x_txt = left + width + (gap if width >= 0 else -gap)
        ha    = "left" if width >= 0 else "right"
        ax.text(x_txt, y_pos[k], f"{width:+.3f}",
                va="center", ha=ha, fontsize=8.5, fontweight="bold")

    # Y-axis: feature name + value, with enough left margin
    y_labels = [f"{fl_ord[k]}  [{fv_ord[k]:.3f}]" for k in range(len(sv_ord))]
    ax.set_yticks(y_pos)
    ax.set_yticklabels(y_labels, fontsize=9)

    # Base value and final prediction lines
    ax.axvline(expected_val, color="gray", linestyle=":", linewidth=1.2, zorder=2)
    ax.axvline(cumsum[-1],   color="black", linestyle="-", linewidth=1.0, zorder=2)

    # f(x) box — placed well above top bar
    ax.text(cumsum[-1], len(sv_ord) + 0.15,
            f"f(x) = {prob:.3f}", ha="center", va="bottom",
            fontsize=9.5, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.3", fc="lightyellow", ec="#888888", alpha=0.95))

    # Base value label — larger font, below bottom bar
    ax.text(expected_val, -1.1,
            f"E[f(x)] = {expected_val:.3f}",
            ha="center", va="top", fontsize=9, color="#555555",
            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="lightgray", alpha=0.8))

    ax.set_xlabel("SHAP Value (log-odds)", fontsize=10)
    ax.set_title(meta["title"], fontsize=12, fontweight="bold", pad=10)
    ax.set_facecolor("white")
    ax.grid(axis="x", linestyle="--", alpha=0.3, zorder=0)
    ax.spines[["top","right"]].set_visible(False)
    ax.set_ylim(-1.6, len(sv_ord) + 0.7)

# Legend
pos_patch = mpatches.Patch(color="#E41A1C", alpha=0.88, label="Increases pathogenicity score")
neg_patch = mpatches.Patch(color="#377EB8", alpha=0.88, label="Decreases pathogenicity score")
fig.legend(handles=[pos_patch, neg_patch], loc="lower center",
           ncol=2, fontsize=10.5, framealpha=0.9,
           bbox_to_anchor=(0.5, 0.01), edgecolor="lightgray")

fig.suptitle("SHAP Waterfall — Per-Variant Feature Contributions  |  XGBoost (OpenVariant)\n"
             "Case studies: TP53 R248W · BRCA1 R496H · BRCA2 D2723A  [SIMULATED]",
             fontsize=12.5, fontweight="bold", y=0.98)

plt.savefig("/mnt/results/figures/Figure6.png", dpi=300, bbox_inches="tight",
            facecolor="white", edgecolor="none")
plt.savefig("/mnt/results/figures/Figure6.svg", bbox_inches="tight",
            facecolor="white", edgecolor="none")
plt.close()
print("✓ Figure6.png + Figure6.svg saved (fixed)")


✓ Figure6.png + Figure6.svg saved (fixed)


Figure 6 final fix — separate y-labels from bar annotations using wider figure and left-aligned labels

In [37]:

import pickle
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

with open("/workspace/artifacts.pkl", "rb") as f:
    art = pickle.load(f)

shap_cases   = art["shap_cases"]
expected_val = art["expected_val"]
FEATURES_OV  = art["FEATURES_OV"]
df_cases     = art["df_cases"]
xgb_final    = art["xgb_final"]

feat_labels = [
    "SIFT Score",
    "PolyPhen-2",
    "gnomAD AF",
    "CADD Phred",
    "REVEL",
    "Funct. Domain",
    "Conservation",
    "Splice Prox.",
]

case_meta = [
    {"title": "TP53 p.R248W  (Pathogenic)"},
    {"title": "BRCA1 p.R496H  (Pathogenic)"},
    {"title": "BRCA2 p.D2723A  (Benign)"},
]

X_cases    = df_cases[FEATURES_OV].values
pred_probs = xgb_final.predict_proba(X_cases)[:, 1]

fig, axes = plt.subplots(1, 3, figsize=(20, 8), facecolor="white")
fig.subplots_adjust(wspace=0.65, top=0.87, bottom=0.14, left=0.06, right=0.97)

for ax_idx, (ax, meta) in enumerate(zip(axes, case_meta)):
    sv   = shap_cases[ax_idx]
    fv   = X_cases[ax_idx]
    prob = pred_probs[ax_idx]

    order   = np.argsort(np.abs(sv))
    sv_ord  = sv[order]
    fv_ord  = fv[order]
    fl_ord  = [feat_labels[i] for i in order]

    cumsum    = np.zeros(len(sv_ord) + 1)
    cumsum[0] = expected_val
    for k in range(len(sv_ord)):
        cumsum[k + 1] = cumsum[k] + sv_ord[k]

    y_pos = np.arange(len(sv_ord))

    for k in range(len(sv_ord)):
        left  = cumsum[k]
        width = sv_ord[k]
        color = "#E41A1C" if width >= 0 else "#377EB8"
        ax.barh(y_pos[k], width, left=left, height=0.50,
                color=color, alpha=0.88, zorder=3)

        # SHAP annotation — place at end of bar with white background box
        gap   = 0.015
        x_txt = left + width + (gap if width >= 0 else -gap)
        ha    = "left" if width >= 0 else "right"
        ax.text(x_txt, y_pos[k], f"{width:+.3f}",
                va="center", ha=ha, fontsize=8, fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.1", fc="white", ec="none", alpha=0.85))

    # Y-axis: feature name only (value on separate line inside bar area)
    ax.set_yticks(y_pos)
    ax.set_yticklabels([f"{fl_ord[k]}" for k in range(len(sv_ord))], fontsize=9.5)

    # Feature value as small text inside/near bar start
    for k in range(len(sv_ord)):
        fval_str = f"={fv_ord[k]:.3f}"
        ax.text(expected_val + 0.01, y_pos[k] - 0.28,
                fval_str, va="center", ha="left", fontsize=7.5,
                color="#555555", style="italic")

    # Reference lines
    ax.axvline(expected_val, color="gray", linestyle=":", linewidth=1.2, zorder=2)
    ax.axvline(cumsum[-1],   color="black", linestyle="-", linewidth=1.0, zorder=2)

    # f(x) box — well above top bar
    ax.text(cumsum[-1], len(sv_ord) + 0.3,
            f"f(x) = {prob:.3f}", ha="center", va="bottom",
            fontsize=10, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.3", fc="lightyellow", ec="#888888", alpha=0.95))

    # Base value label
    ax.text(expected_val, -1.3,
            f"E[f(x)] = {expected_val:.3f}",
            ha="center", va="top", fontsize=9, color="#555555",
            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="lightgray", alpha=0.85))

    ax.set_xlabel("SHAP Value (log-odds)", fontsize=10)
    ax.set_title(meta["title"], fontsize=11.5, fontweight="bold", pad=12)
    ax.set_facecolor("white")
    ax.grid(axis="x", linestyle="--", alpha=0.3, zorder=0)
    ax.spines[["top","right"]].set_visible(False)
    ax.set_ylim(-1.8, len(sv_ord) + 0.9)

# Legend
pos_patch = mpatches.Patch(color="#E41A1C", alpha=0.88, label="Increases pathogenicity score")
neg_patch = mpatches.Patch(color="#377EB8", alpha=0.88, label="Decreases pathogenicity score")
fig.legend(handles=[pos_patch, neg_patch], loc="lower center",
           ncol=2, fontsize=10.5, framealpha=0.9,
           bbox_to_anchor=(0.5, 0.01), edgecolor="lightgray")

fig.suptitle("SHAP Waterfall — Per-Variant Feature Contributions  |  XGBoost (OpenVariant)\n"
             "Case studies: TP53 R248W · BRCA1 R496H · BRCA2 D2723A  [SIMULATED DATA]",
             fontsize=12.5, fontweight="bold", y=0.98)

plt.savefig("/mnt/results/figures/Figure6.png", dpi=300, bbox_inches="tight",
            facecolor="white", edgecolor="none")
plt.savefig("/mnt/results/figures/Figure6.svg", bbox_inches="tight",
            facecolor="white", edgecolor="none")
plt.close()
print("✓ Figure6.png + Figure6.svg saved (final)")


✓ Figure6.png + Figure6.svg saved (final)


Figure 6 final — clean layout with SHAP values inside bars, no label collisions

In [39]:

import pickle
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

with open("/workspace/artifacts.pkl", "rb") as f:
    art = pickle.load(f)

shap_cases   = art["shap_cases"]
expected_val = art["expected_val"]
FEATURES_OV  = art["FEATURES_OV"]
df_cases     = art["df_cases"]
xgb_final    = art["xgb_final"]

feat_labels = [
    "SIFT",
    "PolyPhen-2",
    "gnomAD AF",
    "CADD",
    "REVEL",
    "Domain",
    "Conservation",
    "Splice",
]

case_titles = [
    "TP53 p.R248W\n(Pathogenic)",
    "BRCA1 p.R496H\n(Pathogenic)",
    "BRCA2 p.D2723A\n(Benign)",
]

X_cases    = df_cases[FEATURES_OV].values
pred_probs = xgb_final.predict_proba(X_cases)[:, 1]

fig, axes = plt.subplots(1, 3, figsize=(18, 7), facecolor="white")
fig.subplots_adjust(wspace=0.50, top=0.85, bottom=0.14, left=0.07, right=0.97)

for ax_idx, ax in enumerate(axes):
    sv   = shap_cases[ax_idx]
    fv   = X_cases[ax_idx]
    prob = pred_probs[ax_idx]

    order   = np.argsort(np.abs(sv))
    sv_ord  = sv[order]
    fv_ord  = fv[order]
    fl_ord  = [feat_labels[i] for i in order]

    cumsum    = np.zeros(len(sv_ord) + 1)
    cumsum[0] = expected_val
    for k in range(len(sv_ord)):
        cumsum[k + 1] = cumsum[k] + sv_ord[k]

    y_pos = np.arange(len(sv_ord))

    for k in range(len(sv_ord)):
        left  = cumsum[k]
        width = sv_ord[k]
        color = "#E41A1C" if width >= 0 else "#377EB8"
        ax.barh(y_pos[k], width, left=left, height=0.52,
                color=color, alpha=0.85, zorder=3)

        # SHAP value: inside bar if wide enough, else outside with white bg
        bar_center = left + width / 2
        if abs(width) > 0.12:
            ax.text(bar_center, y_pos[k], f"{width:+.3f}",
                    va="center", ha="center", fontsize=8,
                    color="white", fontweight="bold", zorder=5)
        else:
            x_out = left + width + (0.02 if width >= 0 else -0.02)
            ha_out = "left" if width >= 0 else "right"
            ax.text(x_out, y_pos[k], f"{width:+.3f}",
                    va="center", ha=ha_out, fontsize=7.5,
                    color="black", fontweight="bold", zorder=5,
                    bbox=dict(boxstyle="round,pad=0.08", fc="white", ec="none", alpha=0.8))

    # Y-axis: clean feature names only
    ax.set_yticks(y_pos)
    ax.set_yticklabels(fl_ord, fontsize=10.5)

    # Feature values as a right-side annotation column
    x_right = ax.get_xlim()[1] if ax_idx == 0 else None
    for k in range(len(sv_ord)):
        fval = fv_ord[k]
        fval_str = f"{fval:.3f}" if isinstance(fval, float) else str(int(fval))
        ax.annotate(fval_str,
                    xy=(1.01, (y_pos[k] + 0.5) / len(sv_ord)),
                    xycoords=("axes fraction", "axes fraction"),
                    fontsize=8, color="#666666", va="center", ha="left",
                    annotation_clip=False)

    # Reference lines
    ax.axvline(expected_val, color="#888888", linestyle=":", linewidth=1.2, zorder=2)
    ax.axvline(cumsum[-1],   color="black",   linestyle="-", linewidth=1.0, zorder=2)

    # f(x) box — above top bar, well clear of title
    ax.text(cumsum[-1], len(sv_ord) - 0.05,
            f"f(x) = {prob:.3f}", ha="center", va="bottom",
            fontsize=10, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.3", fc="lightyellow", ec="#888888", alpha=0.95),
            zorder=6)

    # Base value label below bottom bar
    ax.text(expected_val, -0.75,
            f"E[f(x)]={expected_val:.3f}",
            ha="center", va="top", fontsize=8.5, color="#555555",
            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="lightgray", alpha=0.85))

    ax.set_xlabel("SHAP Value (log-odds)", fontsize=10)
    ax.set_title(case_titles[ax_idx], fontsize=11.5, fontweight="bold", pad=10)
    ax.set_facecolor("white")
    ax.grid(axis="x", linestyle="--", alpha=0.3, zorder=0)
    ax.spines[["top","right"]].set_visible(False)
    ax.set_ylim(-1.2, len(sv_ord) + 0.3)

    # Column header for feature values
    ax.text(1.01, 1.01, "Val.", transform=ax.transAxes,
            fontsize=8, color="#666666", ha="left", va="bottom",
            annotation_clip=False)

# Legend
pos_patch = mpatches.Patch(color="#E41A1C", alpha=0.85, label="Increases pathogenicity")
neg_patch = mpatches.Patch(color="#377EB8", alpha=0.85, label="Decreases pathogenicity")
fig.legend(handles=[pos_patch, neg_patch], loc="lower center",
           ncol=2, fontsize=10.5, framealpha=0.9,
           bbox_to_anchor=(0.5, 0.01), edgecolor="lightgray")

fig.suptitle("SHAP Waterfall — Per-Variant Feature Contributions  |  XGBoost (OpenVariant)\n"
             "Case studies: TP53 R248W · BRCA1 R496H · BRCA2 D2723A  [SIMULATED DATA]",
             fontsize=12.5, fontweight="bold", y=0.97)

plt.savefig("/mnt/results/figures/Figure6.png", dpi=300, bbox_inches="tight",
            facecolor="white", edgecolor="none")
plt.savefig("/mnt/results/figures/Figure6.svg", bbox_inches="tight",
            facecolor="white", edgecolor="none")
plt.close()
print("✓ Figure6.png + Figure6.svg saved (clean layout)")


Traceback (most recent call last):
  File "<string>", line 71, in <module>
  File "<string>", line 119, in <module>
  File "/opt/conda/lib/python3.11/site-packages/matplotlib/axes/_axes.py", line 710, in text
    t = mtext.Text(x, y, text=s, **effective_kwargs)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/matplotlib/text.py", line 155, in __init__
    self.update(kwargs)
  File "/opt/conda/lib/python3.11/site-packages/matplotlib/text.py", line 205, in update
    ret.extend(super().update(kwargs))
               ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/matplotlib/artist.py", line 1223, in update
    return self._update_props(
           ^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/matplotlib/artist.py", line 1206, in _update_props
    raise AttributeError(
AttributeError: 'Text' object has no property 'annotation_clip'


Figure 6 final — fixed annotation_clip error, clean no-collision layout

In [41]:

import pickle
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.transforms import blended_transform_factory

with open("/workspace/artifacts.pkl", "rb") as f:
    art = pickle.load(f)

shap_cases   = art["shap_cases"]
expected_val = art["expected_val"]
FEATURES_OV  = art["FEATURES_OV"]
df_cases     = art["df_cases"]
xgb_final    = art["xgb_final"]

feat_labels = ["SIFT", "PolyPhen-2", "gnomAD AF", "CADD",
               "REVEL", "Domain", "Conservation", "Splice"]

case_titles = [
    "TP53 p.R248W\n(Pathogenic)",
    "BRCA1 p.R496H\n(Pathogenic)",
    "BRCA2 p.D2723A\n(Benign)",
]

X_cases    = df_cases[FEATURES_OV].values
pred_probs = xgb_final.predict_proba(X_cases)[:, 1]

fig, axes = plt.subplots(1, 3, figsize=(18, 7), facecolor="white")
fig.subplots_adjust(wspace=0.55, top=0.85, bottom=0.14, left=0.06, right=0.95)

for ax_idx, ax in enumerate(axes):
    sv   = shap_cases[ax_idx]
    fv   = X_cases[ax_idx]
    prob = pred_probs[ax_idx]

    order   = np.argsort(np.abs(sv))
    sv_ord  = sv[order]
    fv_ord  = fv[order]
    fl_ord  = [feat_labels[i] for i in order]

    cumsum    = np.zeros(len(sv_ord) + 1)
    cumsum[0] = expected_val
    for k in range(len(sv_ord)):
        cumsum[k + 1] = cumsum[k] + sv_ord[k]

    y_pos = np.arange(len(sv_ord))

    # ── Draw bars ──────────────────────────────────────────────────────────────
    for k in range(len(sv_ord)):
        left  = cumsum[k]
        width = sv_ord[k]
        color = "#E41A1C" if width >= 0 else "#377EB8"
        ax.barh(y_pos[k], width, left=left, height=0.52,
                color=color, alpha=0.85, zorder=3)

        # SHAP value: inside bar if wide enough, else outside
        if abs(width) > 0.12:
            ax.text(left + width / 2, y_pos[k], f"{width:+.3f}",
                    va="center", ha="center", fontsize=8,
                    color="white", fontweight="bold", zorder=5)
        else:
            x_out = left + width + (0.02 if width >= 0 else -0.02)
            ha_out = "left" if width >= 0 else "right"
            ax.text(x_out, y_pos[k], f"{width:+.3f}",
                    va="center", ha=ha_out, fontsize=7.5,
                    color="black", fontweight="bold", zorder=5,
                    bbox=dict(boxstyle="round,pad=0.08", fc="white", ec="none", alpha=0.8))

    # ── Y-axis: clean feature names only ──────────────────────────────────────
    ax.set_yticks(y_pos)
    ax.set_yticklabels(fl_ord, fontsize=10.5)

    # ── Feature values: right-side column via ax.annotate (clip=False) ────────
    # Use axes-fraction x, data y
    trans = blended_transform_factory(ax.transAxes, ax.transData)
    for k in range(len(sv_ord)):
        fval = fv_ord[k]
        fval_str = f"{fval:.3f}" if not float(fval).is_integer() else str(int(fval))
        ax.text(1.03, y_pos[k], fval_str,
                transform=trans,
                fontsize=8, color="#666666", va="center", ha="left",
                clip_on=False)

    # Column header
    ax.text(1.03, len(sv_ord) - 0.3, "Val.",
            transform=trans,
            fontsize=8, color="#444444", va="center", ha="left",
            fontweight="bold", clip_on=False)

    # ── Reference lines ───────────────────────────────────────────────────────
    ax.axvline(expected_val, color="#888888", linestyle=":", linewidth=1.2, zorder=2)
    ax.axvline(cumsum[-1],   color="black",   linestyle="-",  linewidth=1.0, zorder=2)

    # ── f(x) box — above top bar ──────────────────────────────────────────────
    ax.text(cumsum[-1], len(sv_ord) - 0.1,
            f"f(x) = {prob:.3f}", ha="center", va="bottom",
            fontsize=10, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.3", fc="lightyellow", ec="#888888", alpha=0.95),
            zorder=6)

    # ── Base value label ──────────────────────────────────────────────────────
    ax.text(expected_val, -0.75,
            f"E[f(x)]={expected_val:.3f}",
            ha="center", va="top", fontsize=8.5, color="#555555",
            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="lightgray", alpha=0.85))

    ax.set_xlabel("SHAP Value (log-odds)", fontsize=10)
    ax.set_title(case_titles[ax_idx], fontsize=11.5, fontweight="bold", pad=10)
    ax.set_facecolor("white")
    ax.grid(axis="x", linestyle="--", alpha=0.3, zorder=0)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_ylim(-1.2, len(sv_ord) + 0.3)

# ── Legend ────────────────────────────────────────────────────────────────────
pos_patch = mpatches.Patch(color="#E41A1C", alpha=0.85, label="Increases pathogenicity")
neg_patch = mpatches.Patch(color="#377EB8", alpha=0.85, label="Decreases pathogenicity")
fig.legend(handles=[pos_patch, neg_patch], loc="lower center",
           ncol=2, fontsize=10.5, framealpha=0.9,
           bbox_to_anchor=(0.5, 0.01), edgecolor="lightgray")

fig.suptitle(
    "SHAP Waterfall — Per-Variant Feature Contributions  |  XGBoost (OpenVariant)\n"
    "Case studies: TP53 R248W · BRCA1 R496H · BRCA2 D2723A  [SIMULATED DATA]",
    fontsize=12.5, fontweight="bold", y=0.97)

plt.savefig("/mnt/results/figures/Figure6.png", dpi=300, bbox_inches="tight",
            facecolor="white", edgecolor="none")
plt.savefig("/mnt/results/figures/Figure6.svg", bbox_inches="tight",
            facecolor="white", edgecolor="none")
plt.close()
print("✓ Figure6.png + Figure6.svg saved")


✓ Figure6.png + Figure6.svg saved


Figure 6 definitive — fixed x-range, all annotations in dedicated columns, no collisions

In [43]:

import pickle
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.transforms import blended_transform_factory

with open("/workspace/artifacts.pkl", "rb") as f:
    art = pickle.load(f)

shap_cases   = art["shap_cases"]
expected_val = art["expected_val"]
FEATURES_OV  = art["FEATURES_OV"]
df_cases     = art["df_cases"]
xgb_final    = art["xgb_final"]

feat_labels = ["SIFT", "PolyPhen-2", "gnomAD AF", "CADD",
               "REVEL", "Domain", "Conservation", "Splice"]

case_titles = [
    "TP53 p.R248W  (Pathogenic)",
    "BRCA1 p.R496H  (Pathogenic)",
    "BRCA2 p.D2723A  (Benign)",
]

X_cases    = df_cases[FEATURES_OV].values
pred_probs = xgb_final.predict_proba(X_cases)[:, 1]

# ── Layout: 3 panels, each panel = [y-labels | bars | shap-vals | feat-vals] ──
fig, axes = plt.subplots(1, 3, figsize=(20, 8), facecolor="white")
fig.subplots_adjust(wspace=0.08, top=0.84, bottom=0.12, left=0.02, right=0.98)

for ax_idx, ax in enumerate(axes):
    sv   = shap_cases[ax_idx]
    fv   = X_cases[ax_idx]
    prob = pred_probs[ax_idx]

    order   = np.argsort(np.abs(sv))
    sv_ord  = sv[order]
    fv_ord  = fv[order]
    fl_ord  = [feat_labels[i] for i in order]
    n       = len(sv_ord)

    cumsum    = np.zeros(n + 1)
    cumsum[0] = expected_val
    for k in range(n):
        cumsum[k + 1] = cumsum[k] + sv_ord[k]

    y_pos = np.arange(n, dtype=float)

    # Determine symmetric x-range with padding
    all_x = list(cumsum)
    x_min = min(all_x) - 0.15
    x_max = max(all_x) + 0.15
    x_range = max(abs(x_min - expected_val), abs(x_max - expected_val)) * 1.3
    ax.set_xlim(expected_val - x_range, expected_val + x_range)

    # ── Bars ──────────────────────────────────────────────────────────────────
    for k in range(n):
        left  = cumsum[k]
        width = sv_ord[k]
        color = "#E41A1C" if width >= 0 else "#377EB8"
        ax.barh(y_pos[k], width, left=left, height=0.50,
                color=color, alpha=0.85, zorder=3)

    # ── SHAP value annotations: always in a fixed right column ────────────────
    trans_mixed = blended_transform_factory(ax.transAxes, ax.transData)
    # Right column at axes x=1.01 (outside plot)
    for k in range(n):
        ax.text(1.02, y_pos[k], f"{sv_ord[k]:+.3f}",
                transform=trans_mixed,
                fontsize=9, color="#E41A1C" if sv_ord[k] >= 0 else "#377EB8",
                va="center", ha="left", fontweight="bold", clip_on=False)

    # Feature value column at axes x=1.14
    for k in range(n):
        fval = fv_ord[k]
        fval_str = f"{fval:.3f}" if not float(fval).is_integer() else str(int(fval))
        ax.text(1.14, y_pos[k], fval_str,
                transform=trans_mixed,
                fontsize=8.5, color="#555555",
                va="center", ha="left", clip_on=False)

    # Column headers
    ax.text(1.02, n - 0.4, "SHAP",
            transform=trans_mixed, fontsize=8, color="#333333",
            va="center", ha="left", fontweight="bold", clip_on=False)
    ax.text(1.14, n - 0.4, "Val.",
            transform=trans_mixed, fontsize=8, color="#333333",
            va="center", ha="left", fontweight="bold", clip_on=False)

    # ── Y-axis: feature names only, no values ─────────────────────────────────
    ax.set_yticks(y_pos)
    ax.set_yticklabels(fl_ord, fontsize=10.5)
    ax.tick_params(axis="y", pad=4)

    # ── Reference lines ───────────────────────────────────────────────────────
    ax.axvline(expected_val, color="#888888", linestyle=":", linewidth=1.3, zorder=2)
    ax.axvline(cumsum[-1],   color="black",   linestyle="-",  linewidth=1.0, zorder=2)

    # ── f(x) box — top of plot, centered on final cumsum ─────────────────────
    ax.text(cumsum[-1], n + 0.05,
            f"f(x) = {prob:.3f}", ha="center", va="bottom",
            fontsize=10.5, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.3", fc="lightyellow", ec="#888888", alpha=0.95),
            zorder=6)

    # ── Base value label ──────────────────────────────────────────────────────
    ax.text(expected_val, -0.85,
            f"E[f(x)] = {expected_val:.3f}",
            ha="center", va="top", fontsize=9, color="#555555",
            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="lightgray", alpha=0.85))

    ax.set_xlabel("SHAP Value (log-odds)", fontsize=10)
    ax.set_title(case_titles[ax_idx], fontsize=11.5, fontweight="bold", pad=10)
    ax.set_facecolor("white")
    ax.grid(axis="x", linestyle="--", alpha=0.3, zorder=0)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_ylim(-1.3, n + 0.5)

# ── Legend ────────────────────────────────────────────────────────────────────
pos_patch = mpatches.Patch(color="#E41A1C", alpha=0.85, label="Increases pathogenicity")
neg_patch = mpatches.Patch(color="#377EB8", alpha=0.85, label="Decreases pathogenicity")
fig.legend(handles=[pos_patch, neg_patch], loc="lower center",
           ncol=2, fontsize=10.5, framealpha=0.9,
           bbox_to_anchor=(0.5, 0.01), edgecolor="lightgray")

fig.suptitle(
    "SHAP Waterfall — Per-Variant Feature Contributions  |  XGBoost (OpenVariant)\n"
    "Case studies: TP53 R248W · BRCA1 R496H · BRCA2 D2723A  [SIMULATED DATA]",
    fontsize=12.5, fontweight="bold", y=0.97)

plt.savefig("/mnt/results/figures/Figure6.png", dpi=300, bbox_inches="tight",
            facecolor="white", edgecolor="none")
plt.savefig("/mnt/results/figures/Figure6.svg", bbox_inches="tight",
            facecolor="white", edgecolor="none")
plt.close()
print("✓ Figure6.png + Figure6.svg saved (definitive)")


✓ Figure6.png + Figure6.svg saved (definitive)


Figure 6 — clean waterfall using GridSpec with dedicated text axes for labels

In [45]:

import pickle
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec

with open("/workspace/artifacts.pkl", "rb") as f:
    art = pickle.load(f)

shap_cases   = art["shap_cases"]
expected_val = art["expected_val"]
FEATURES_OV  = art["FEATURES_OV"]
df_cases     = art["df_cases"]
xgb_final    = art["xgb_final"]

feat_labels_full = ["SIFT Score", "PolyPhen-2", "gnomAD AF", "CADD Phred",
                    "REVEL", "Funct. Domain", "Conservation", "Splice Prox."]

case_titles = [
    "TP53 p.R248W  (Pathogenic)",
    "BRCA1 p.R496H  (Pathogenic)",
    "BRCA2 p.D2723A  (Benign)",
]

X_cases    = df_cases[FEATURES_OV].values
pred_probs = xgb_final.predict_proba(X_cases)[:, 1]

# Use consistent feature order across all panels (sorted by mean |SHAP|)
mean_abs = np.abs(shap_cases).mean(axis=0)
global_order = np.argsort(mean_abs)   # ascending = bottom to top

fig = plt.figure(figsize=(20, 8), facecolor="white")
outer = gridspec.GridSpec(1, 3, figure=fig, wspace=0.45,
                          left=0.04, right=0.97, top=0.84, bottom=0.13)

n = len(global_order)
y_pos = np.arange(n, dtype=float)

for ax_idx in range(3):
    sv   = shap_cases[ax_idx][global_order]
    fv   = X_cases[ax_idx][global_order]
    fl   = [feat_labels_full[i] for i in global_order]
    prob = pred_probs[ax_idx]

    cumsum    = np.zeros(n + 1)
    cumsum[0] = expected_val
    for k in range(n):
        cumsum[k + 1] = cumsum[k] + sv[k]

    # Inner GridSpec: [label_col | bar_col | shap_col | val_col]
    inner = gridspec.GridSpecFromSubplotSpec(
        1, 4, subplot_spec=outer[ax_idx],
        width_ratios=[2.2, 5, 1.4, 1.2], wspace=0
    )

    ax_lbl  = fig.add_subplot(inner[0])   # feature names
    ax_bar  = fig.add_subplot(inner[1])   # waterfall bars
    ax_shap = fig.add_subplot(inner[2])   # SHAP values
    ax_val  = fig.add_subplot(inner[3])   # feature values

    # ── Feature name column ───────────────────────────────────────────────────
    for k in range(n):
        ax_lbl.text(0.98, y_pos[k], fl[k],
                    ha="right", va="center", fontsize=9.5, transform=ax_lbl.transData)
    ax_lbl.set_xlim(0, 1)
    ax_lbl.set_ylim(-1.3, n + 0.5)
    ax_lbl.axis("off")

    # ── Bar column ────────────────────────────────────────────────────────────
    x_range = max(abs(min(cumsum) - expected_val),
                  abs(max(cumsum) - expected_val)) * 1.25 + 0.05
    ax_bar.set_xlim(expected_val - x_range, expected_val + x_range)
    ax_bar.set_ylim(-1.3, n + 0.5)

    for k in range(n):
        left  = cumsum[k]
        width = sv[k]
        color = "#E41A1C" if width >= 0 else "#377EB8"
        ax_bar.barh(y_pos[k], width, left=left, height=0.50,
                    color=color, alpha=0.85, zorder=3)

    ax_bar.axvline(expected_val, color="#888888", linestyle=":", linewidth=1.2, zorder=2)
    ax_bar.axvline(cumsum[-1],   color="black",   linestyle="-",  linewidth=1.0, zorder=2)

    # f(x) box
    ax_bar.text(cumsum[-1], n + 0.05,
                f"f(x) = {prob:.3f}", ha="center", va="bottom",
                fontsize=10, fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.3", fc="lightyellow", ec="#888888", alpha=0.95),
                zorder=6)

    # Base value
    ax_bar.text(expected_val, -0.9,
                f"E[f(x)]={expected_val:.3f}",
                ha="center", va="top", fontsize=8.5, color="#555555",
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="lightgray", alpha=0.85))

    ax_bar.set_yticks([])
    ax_bar.set_xlabel("SHAP Value (log-odds)", fontsize=9.5)
    ax_bar.set_facecolor("white")
    ax_bar.grid(axis="x", linestyle="--", alpha=0.3, zorder=0)
    ax_bar.spines[["top", "right", "left"]].set_visible(False)
    ax_bar.set_title(case_titles[ax_idx], fontsize=11, fontweight="bold", pad=10)

    # ── SHAP value column ─────────────────────────────────────────────────────
    ax_shap.set_xlim(0, 1)
    ax_shap.set_ylim(-1.3, n + 0.5)
    ax_shap.text(0.5, n - 0.4, "SHAP", ha="center", va="center",
                 fontsize=8, fontweight="bold", color="#333333")
    for k in range(n):
        color = "#E41A1C" if sv[k] >= 0 else "#377EB8"
        ax_shap.text(0.5, y_pos[k], f"{sv[k]:+.3f}",
                     ha="center", va="center", fontsize=9,
                     color=color, fontweight="bold")
    ax_shap.axis("off")

    # ── Feature value column ──────────────────────────────────────────────────
    ax_val.set_xlim(0, 1)
    ax_val.set_ylim(-1.3, n + 0.5)
    ax_val.text(0.5, n - 0.4, "Val.", ha="center", va="center",
                fontsize=8, fontweight="bold", color="#333333")
    for k in range(n):
        fval = fv[k]
        fval_str = f"{fval:.3f}" if not float(fval).is_integer() else str(int(fval))
        ax_val.text(0.5, y_pos[k], fval_str,
                    ha="center", va="center", fontsize=8.5, color="#555555")
    ax_val.axis("off")

# ── Legend ────────────────────────────────────────────────────────────────────
pos_patch = mpatches.Patch(color="#E41A1C", alpha=0.85, label="Increases pathogenicity")
neg_patch = mpatches.Patch(color="#377EB8", alpha=0.85, label="Decreases pathogenicity")
fig.legend(handles=[pos_patch, neg_patch], loc="lower center",
           ncol=2, fontsize=10.5, framealpha=0.9,
           bbox_to_anchor=(0.5, 0.01), edgecolor="lightgray")

fig.suptitle(
    "SHAP Waterfall — Per-Variant Feature Contributions  |  XGBoost (OpenVariant)\n"
    "Case studies: TP53 R248W · BRCA1 R496H · BRCA2 D2723A  [SIMULATED DATA]",
    fontsize=12.5, fontweight="bold", y=0.97)

plt.savefig("/mnt/results/figures/Figure6.png", dpi=300, bbox_inches="tight",
            facecolor="white", edgecolor="none")
plt.savefig("/mnt/results/figures/Figure6.svg", bbox_inches="tight",
            facecolor="white", edgecolor="none")
plt.close()
print("✓ Figure6.png + Figure6.svg saved (GridSpec layout)")


✓ Figure6.png + Figure6.svg saved (GridSpec layout)


Extract and print actual correlation matrix and SHAP rankings to verify report text

In [47]:

import pickle
import numpy as np
import pandas as pd

with open("/workspace/artifacts.pkl", "rb") as f:
    art = pickle.load(f)

df          = art["df"]
shap_values = art["shap_values"]
shap_cases  = art["shap_cases"]
FEATURES_OV = art["FEATURES_OV"]
df_cases    = art["df_cases"]

feat_labels = {
    "sift_score":             "SIFT Score",
    "polyphen2_score":        "PolyPhen-2",
    "gnomad_af":              "gnomAD AF",
    "cadd_phred":             "CADD Phred",
    "revel_score":            "REVEL",
    "in_functional_domain":   "Funct. Domain",
    "conservation_score":     "Conservation",
    "splice_proximity":       "Splice Prox.",
}

# ── 1. Actual correlation matrix ──────────────────────────────────────────────
all_feats  = FEATURES_OV + ["alphamissense_score"]
all_labels = [feat_labels.get(f, f) for f in FEATURES_OV] + ["AlphaMissense"]
corr = df[all_feats].corr(method="pearson")
corr.index   = all_labels
corr.columns = all_labels

print("── Actual Pearson Correlation Matrix ──────────────────────────────")
print(corr.round(3).to_string())

# Key pairs for report
print("\n── Key correlation pairs ──────────────────────────────────────────")
pairs = [
    ("REVEL", "PolyPhen-2"),
    ("SIFT Score", "PolyPhen-2"),
    ("gnomAD AF", "CADD Phred"),
    ("AlphaMissense", "REVEL"),
    ("AlphaMissense", "PolyPhen-2"),
    ("gnomAD AF", "PolyPhen-2"),
]
for a, b in pairs:
    print(f"  {a:20s} vs {b:20s}: r = {corr.loc[a, b]:.3f}")

# ── 2. Mean |SHAP| per feature ────────────────────────────────────────────────
mean_abs = np.abs(shap_values).mean(axis=0)
shap_rank = sorted(zip([feat_labels[f] for f in FEATURES_OV], mean_abs),
                   key=lambda x: -x[1])
print("\n── Mean |SHAP| per feature (XGBoost, full training set) ───────────")
for rank, (name, val) in enumerate(shap_rank, 1):
    print(f"  {rank}. {name:22s}: {val:.4f}")

# ── 3. Per-case SHAP values ───────────────────────────────────────────────────
case_names = ["TP53 p.R248W", "BRCA1 p.R496H", "BRCA2 p.D2723A"]
global_order = np.argsort(mean_abs)   # ascending = bottom to top

print("\n── Per-case SHAP values (sorted by |SHAP|, ascending) ─────────────")
for ci, cname in enumerate(case_names):
    sv = shap_cases[ci]
    print(f"\n  {cname}:")
    for fi in reversed(global_order):
        fname = feat_labels[FEATURES_OV[fi]]
        print(f"    {fname:22s}: {sv[fi]:+.4f}")


── Actual Pearson Correlation Matrix ──────────────────────────────
               SIFT Score  PolyPhen-2  gnomAD AF  CADD Phred  REVEL  Funct. Domain  Conservation  Splice Prox.  AlphaMissense
SIFT Score          1.000      -0.787      0.336      -0.742 -0.786         -0.426        -0.686        -0.153         -0.511
PolyPhen-2         -0.787       1.000     -0.317       0.734  0.786          0.446         0.707         0.145          0.531
gnomAD AF           0.336      -0.317      1.000      -0.318 -0.322         -0.158        -0.298        -0.060         -0.168
CADD Phred         -0.742       0.734     -0.318       1.000  0.727          0.407         0.657         0.160          0.473
REVEL              -0.786       0.786     -0.322       0.727  1.000          0.417         0.708         0.144          0.506
Funct. Domain      -0.426       0.446     -0.158       0.407  0.417          1.000         0.379         0.117          0.269
Conservation       -0.686       0.707     -0.298  

Bootstrap per-gene AUC (n=1000) and regenerate Figure 5 with error bars

In [1]:

import pickle
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

with open("/workspace/artifacts.pkl", "rb") as f:
    art = pickle.load(f)

df         = art["df"]
xgb_probs  = art["xgb_probs"]
y          = art["y"]
overall_auc = art["roc_data"]["XGBoost"]["auc"]

genes = ["BRCA1", "BRCA2", "TP53", "TREM2", "EGFR", "ALK"]
rng   = np.random.default_rng(42)
N_BOOT = 1000

gene_results = {}

print("── Per-gene AUC with 95% Bootstrap CI (n=1000) ────────────────────")
for gene in genes:
    mask   = (df["gene"] == gene).values
    y_g    = y[mask]
    p_g    = xgb_probs[mask]
    n_g    = mask.sum()
    n_path = y_g.sum()

    # Point estimate
    auc_pt = roc_auc_score(y_g, p_g)

    # Bootstrap
    boot_aucs = []
    for _ in range(N_BOOT):
        idx = rng.integers(0, n_g, size=n_g)
        yb, pb = y_g[idx], p_g[idx]
        if yb.sum() == 0 or yb.sum() == len(yb):
            continue
        boot_aucs.append(roc_auc_score(yb, pb))

    ci_lo = np.percentile(boot_aucs, 2.5)
    ci_hi = np.percentile(boot_aucs, 97.5)
    se    = np.std(boot_aucs)

    gene_results[gene] = {
        "auc": round(auc_pt, 3),
        "ci_lo": round(ci_lo, 3),
        "ci_hi": round(ci_hi, 3),
        "se": round(se, 3),
        "n": n_g,
        "n_path": int(n_path),
    }
    print(f"  {gene:8s}  n={n_g:3d} (path={int(n_path):3d})  "
          f"AUC = {auc_pt:.3f}  (95% CI: {ci_lo:.3f}–{ci_hi:.3f})")

# Overall bootstrap CI
boot_overall = []
n_all = len(y)
for _ in range(N_BOOT):
    idx = rng.integers(0, n_all, size=n_all)
    yb, pb = y[idx], xgb_probs[idx]
    if yb.sum() == 0 or yb.sum() == n_all:
        continue
    boot_overall.append(roc_auc_score(yb, pb))
ov_lo = np.percentile(boot_overall, 2.5)
ov_hi = np.percentile(boot_overall, 97.5)
print(f"\n  Overall   n={n_all} (path={int(y.sum())})  "
      f"AUC = {overall_auc:.3f}  (95% CI: {ov_lo:.3f}–{ov_hi:.3f})")

# ── Regenerate Figure 5 with error bars ───────────────────────────────────────
genes_sorted = sorted(gene_results.keys(),
                      key=lambda g: gene_results[g]["auc"], reverse=True)
aucs   = [gene_results[g]["auc"]   for g in genes_sorted]
ci_los = [gene_results[g]["ci_lo"] for g in genes_sorted]
ci_his = [gene_results[g]["ci_hi"] for g in genes_sorted]
errs_lo = [aucs[i] - ci_los[i] for i in range(len(genes_sorted))]
errs_hi = [ci_his[i] - aucs[i] for i in range(len(genes_sorted))]

CB_SET2 = ["#66C2A5","#FC8D62","#8DA0CB","#E78AC3","#A6D854","#FFD92F"]

fig, ax = plt.subplots(figsize=(10, 6), facecolor="white")
ax.set_facecolor("white")

bars = ax.bar(genes_sorted, aucs, color=CB_SET2[:len(genes_sorted)],
              width=0.55, zorder=3, edgecolor="white", linewidth=0.8)

# Error bars (95% CI)
ax.errorbar(genes_sorted, aucs,
            yerr=[errs_lo, errs_hi],
            fmt="none", color="black", capsize=5, capthick=1.5,
            linewidth=1.5, zorder=5)

# Annotate with AUC + CI
for i, gene in enumerate(genes_sorted):
    r = gene_results[gene]
    label = f"{r['auc']:.3f}\n({r['ci_lo']:.3f}–{r['ci_hi']:.3f})"
    ax.text(i, r["ci_hi"] + 0.004, label,
            ha="center", va="bottom", fontsize=8.5, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.8))

# Overall AUC reference line
ax.axhline(overall_auc, color="#E41A1C", linestyle="--", linewidth=1.6,
           label=f"Overall AUC = {overall_auc:.3f} (95% CI: {ov_lo:.3f}–{ov_hi:.3f})",
           zorder=4)

ax.set_ylim(0.82, 1.02)
ax.set_xlabel("Gene", fontsize=12)
ax.set_ylabel("AUC-ROC (5-fold CV)", fontsize=12)
ax.set_title("Per-Gene Classification Performance — XGBoost (OpenVariant)\n"
             "Error bars = 95% Bootstrap CI (n=1,000)  [SIMULATED DATA]",
             fontsize=13, fontweight="bold", pad=10)
ax.legend(fontsize=9.5, framealpha=0.9, edgecolor="lightgray", loc="lower right")
ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
ax.spines[["top","right"]].set_visible(False)
fig.text(0.5, 0.01, "⚠ SIMULATED DATA — not real ClinVar records",
         ha="center", fontsize=8, color="gray", style="italic")

plt.tight_layout()
plt.savefig("/mnt/results/figures/Figure5.png", dpi=300, bbox_inches="tight",
            facecolor="white", edgecolor="none")
plt.savefig("/mnt/results/figures/Figure5.svg", bbox_inches="tight",
            facecolor="white", edgecolor="none")
plt.close()
print("\n✓ Figure5.png + Figure5.svg regenerated with bootstrap CI error bars")

# Save for report update
import json
with open("/workspace/gene_ci_results.json", "w") as f:
    json.dump({"genes": gene_results,
               "overall": {"auc": round(overall_auc,3),
                           "ci_lo": round(ov_lo,3),
                           "ci_hi": round(ov_hi,3)}}, f)


── Per-gene AUC with 95% Bootstrap CI (n=1000) ────────────────────
  BRCA1     n=375 (path=110)  AUC = 0.941  (95% CI: 0.916–0.963)
  BRCA2     n=355 (path=105)  AUC = 0.938  (95% CI: 0.912–0.961)
  TP53      n=315 (path=120)  AUC = 0.948  (95% CI: 0.922–0.968)
  TREM2     n=275 (path= 90)  AUC = 0.932  (95% CI: 0.900–0.959)
  EGFR      n=243 (path= 88)  AUC = 0.950  (95% CI: 0.922–0.972)
  ALK       n=241 (path= 85)  AUC = 0.946  (95% CI: 0.907–0.974)

  Overall   n=1804 (path=598)  AUC = 0.942  (95% CI: 0.931–0.953)

✓ Figure5.png + Figure5.svg regenerated with bootstrap CI error bars
Traceback (most recent call last):
  File "<string>", line 71, in <module>
  File "<string>", line 134, in <module>
  File "/opt/conda/lib/python3.11/json/__init__.py", line 179, in dump
    for chunk in iterable:
  File "/opt/conda/lib/python3.11/json/encoder.py", line 432, in _iterencode
    yield from _iterencode_dict(o, _current_indent_level)
  File "/opt/conda/lib/python3.11/json/encoder.py", line

Add joblib warning note to execution trace notebook

In [3]:

import nbformat

nb_path = "/mnt/results/execution_trace.ipynb"
nb = nbformat.read(nb_path, as_version=4)

note_cell = nbformat.v4.new_markdown_cell(source="""## ⚠️ Known Warning: LightGBM joblib CPU count

During model training, LightGBM emits the following non-fatal warning:

```
UserWarning: Could not find the number of physical cores for the following reason:
'cpu_count' not found in module 'joblib.externals.loky.backend.context'
```

**Impact:** LightGBM likely ran single-threaded (n_jobs=1 effective). This does not affect model correctness or AUC values — only training speed. The warning arises from a version mismatch between LightGBM and the installed joblib backend in this environment.

**Resolution:** Not required for reproducibility. To suppress: `import warnings; warnings.filterwarnings('ignore', category=UserWarning, module='lightgbm')`.
""")

# Insert after the first cell (title/imports) so it's visible early
nb.cells.insert(1, note_cell)

nbformat.write(nb, nb_path)
print(f"Notebook updated: {len(nb.cells)} cells total")
print("Joblib warning note inserted at position 1")


Notebook updated: 52 cells total
Joblib warning note inserted at position 1


Verify all per-gene counts, class balance, and bootstrap CIs from actual CSV + artifacts

In [5]:

import pandas as pd
import numpy as np
import pickle
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

# ── Load CSV ──────────────────────────────────────────────────────────────────
df_all = pd.read_csv("/mnt/results/data/SIMULATED_clinvar2026_variants.csv")
df = df_all[~df_all["variant_id"].str.startswith("CASE")].reset_index(drop=True)

print(f"Total rows in CSV: {len(df_all)}")
print(f"Training rows (non-CASE): {len(df)}")
print(f"\nClass balance:")
vc = df["label"].value_counts().sort_index()
print(f"  Benign (0):     {vc[0]}  ({vc[0]/len(df)*100:.1f}%)")
print(f"  Pathogenic (1): {vc[1]}  ({vc[1]/len(df)*100:.1f}%)")

print(f"\nPer-gene counts:")
gene_stats = df.groupby("gene").agg(
    total=("label", "count"),
    pathogenic=("label", "sum")
).reset_index()
gene_stats["benign"] = gene_stats["total"] - gene_stats["pathogenic"]
print(gene_stats.to_string(index=False))


Total rows in CSV: 1807
Training rows (non-CASE): 1804

Class balance:
  Benign (0):     1206  (66.9%)
  Pathogenic (1): 598  (33.1%)

Per-gene counts:
 gene  total  pathogenic  benign
  ALK    241          85     156
BRCA1    375         110     265
BRCA2    355         105     250
 EGFR    243          88     155
 TP53    315         120     195
TREM2    275          90     185


Recompute per-gene bootstrap CIs from artifacts.pkl

In [7]:

import warnings
warnings.filterwarnings("ignore")

# Load saved artifacts
with open("/workspace/artifacts.pkl", "rb") as f:
    artifacts = pickle.load(f)

xgb_model = artifacts["xgb_model"]

FEATURES = ["sift_score", "polyphen2_score", "gnomad_af",
            "cadd_phred", "revel_score",
            "in_functional_domain", "conservation_score", "splice_proximity"]

X = df[FEATURES].values
y = df["label"].values
genes = df["gene"].values

# Get predicted probabilities from full-data model
proba = xgb_model.predict_proba(X)[:, 1]

# Bootstrap per-gene AUC (n=1000, seed=42)
rng = np.random.default_rng(42)
n_boot = 1000
gene_list = ["EGFR", "TP53", "ALK", "BRCA1", "BRCA2", "TREM2"]

results = {}
for gene in gene_list:
    mask = genes == gene
    y_g = y[mask]
    p_g = proba[mask]
    n_g = mask.sum()
    
    # Point estimate
    auc_point = roc_auc_score(y_g, p_g)
    
    # Bootstrap
    boot_aucs = []
    for _ in range(n_boot):
        idx = rng.integers(0, n_g, size=n_g)
        y_b, p_b = y_g[idx], p_g[idx]
        if len(np.unique(y_b)) < 2:
            continue
        boot_aucs.append(roc_auc_score(y_b, p_b))
    
    ci_lo = np.percentile(boot_aucs, 2.5)
    ci_hi = np.percentile(boot_aucs, 97.5)
    results[gene] = {"auc": auc_point, "ci_lo": ci_lo, "ci_hi": ci_hi,
                     "n_boot_valid": len(boot_aucs)}
    print(f"{gene:6s}  AUC={auc_point:.3f}  95% CI: {ci_lo:.3f}–{ci_hi:.3f}  "
          f"(n={n_g}, n_boot={len(boot_aucs)})")

print("\nAll values verified from actual CSV + artifacts.pkl")


Traceback (most recent call last):
  File "<string>", line 71, in <module>
  File "<string>", line 9, in <module>
KeyError: 'xgb_model'


Inspect artifacts.pkl keys to find correct model key

In [9]:

import pickle

with open("/workspace/artifacts.pkl", "rb") as f:
    artifacts = pickle.load(f)

print("Keys in artifacts.pkl:", list(artifacts.keys()))


Keys in artifacts.pkl: ['df', 'df_full', 'df_cases', 'X', 'y', 'FEATURES_OV', 'xgb_probs', 'lgb_probs', 'am_probs', 'roc_data', 'metrics_df', 'xgb_final', 'explainer', 'shap_values', 'shap_cases', 'expected_val', 'gene_aucs']


Recompute per-gene bootstrap CIs using correct artifact keys

In [11]:

import numpy as np
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings("ignore")

# Pull from artifacts
xgb_probs = artifacts["xgb_probs"]   # CV predicted probabilities (out-of-fold)
y         = artifacts["y"]
df_train  = artifacts["df"]           # training DataFrame (1804 rows)
genes     = df_train["gene"].values

print(f"xgb_probs shape: {xgb_probs.shape}, y shape: {y.shape}, genes shape: {genes.shape}")
print(f"Unique genes: {np.unique(genes)}")

# Verify overall AUC matches reported 0.942
overall_auc = roc_auc_score(y, xgb_probs)
print(f"\nOverall XGBoost AUC (CV): {overall_auc:.4f}  (reported: 0.942)")

# Bootstrap per-gene AUC
rng = np.random.default_rng(42)
n_boot = 1000
gene_list = ["EGFR", "TP53", "ALK", "BRCA1", "BRCA2", "TREM2"]

print("\nPer-gene bootstrap CIs (n=1000, seed=42):")
print(f"{'Gene':6s}  {'N':>4s}  {'Path':>4s}  {'AUC':>5s}  {'95% CI':>15s}  {'n_boot_valid':>12s}")
print("-" * 60)

verified_results = {}
for gene in gene_list:
    mask = genes == gene
    y_g  = y[mask]
    p_g  = xgb_probs[mask]
    n_g  = mask.sum()
    n_path = y_g.sum()

    auc_point = roc_auc_score(y_g, p_g)

    boot_aucs = []
    for _ in range(n_boot):
        idx = rng.integers(0, n_g, size=n_g)
        y_b, p_b = y_g[idx], p_g[idx]
        if len(np.unique(y_b)) < 2:
            continue
        boot_aucs.append(roc_auc_score(y_b, p_b))

    ci_lo = np.percentile(boot_aucs, 2.5)
    ci_hi = np.percentile(boot_aucs, 97.5)
    verified_results[gene] = {
        "auc": auc_point, "ci_lo": ci_lo, "ci_hi": ci_hi,
        "n": int(n_g), "n_path": int(n_path), "n_boot_valid": len(boot_aucs)
    }
    print(f"{gene:6s}  {n_g:4d}  {n_path:4d}  {auc_point:.3f}  "
          f"{ci_lo:.3f}–{ci_hi:.3f}  {len(boot_aucs):12d}")

print("\nSource: xgb_probs (out-of-fold CV probabilities) from artifacts.pkl")
print("        y, gene labels from SIMULATED_clinvar2026_variants.csv (N=1804 training rows)")


xgb_probs shape: (1804,), y shape: (1804,), genes shape: (1804,)
Unique genes: ['ALK' 'BRCA1' 'BRCA2' 'EGFR' 'TP53' 'TREM2']

Overall XGBoost AUC (CV): 0.9417  (reported: 0.942)

Per-gene bootstrap CIs (n=1000, seed=42):
Gene       N  Path    AUC           95% CI  n_boot_valid
------------------------------------------------------------
EGFR     243    88  0.950  0.923–0.972          1000
TP53     315   120  0.948  0.923–0.968          1000
ALK      241    85  0.946  0.908–0.974          1000
BRCA1    375   110  0.941  0.917–0.962          1000
BRCA2    355   105  0.938  0.910–0.962          1000
TREM2    275    90  0.932  0.900–0.961          1000

Source: xgb_probs (out-of-fold CV probabilities) from artifacts.pkl
        y, gene labels from SIMULATED_clinvar2026_variants.csv (N=1804 training rows)
